# Final Adaptive Domain-Aware Prompt Router
## Gemma 4 26B A4B IT — Reproducible Multimodal Inference System

This notebook implements the final inference layer of the project: a **single public inference function** that receives a multimodal multiple-choice problem, resolves its academic domain, selects the empirically preferred prompting strategy, executes the corresponding strategy, and returns the final option letter.

The intended public interface is:

```python
answer = solve(QUESTION, IMAGES, API_KEY)
```

The default domain mode is `api_classifier`, so an ordinary user supplies only the question, image(s), and API key.

### Frozen routing policy

| Domain | Strategy |
|---|---|
| Art and Design | CoT |
| Business | CoT |
| Health and Medicine | Few-shot Complexity |
| Humanities and Social Science | Plan-and-Solve |
| Science | CCoT |
| Tech and Engineering | Plan-and-Solve |

The routing table is frozen before held-out evaluation and is never altered using the incoming answer or post-hoc correctness.


## 1. Reproducibility contract and domain modes

The system supports three domain-resolution modes while keeping the downstream strategy implementations fixed.

- **`api_classifier`** — the same multimodal Gemma endpoint is called once to predict a closed-set domain and subject before prompt routing. This is the default end-user mode.
- **`trained_classifier`** — a separately trained local classifier supplies the domain and subject through a stable `predict(...)` interface. No API call is used for domain resolution.
- **`mmmu_pro`** — a pinned MMMU-Pro row is resolved by exact sample ID; its published subject metadata is deterministically mapped to the six project domains.

Every result records `domain_mode` and `domain_source`, so routing-policy evaluation and fully automatic end-to-end evaluation remain distinguishable.

The following components are frozen in code: the six-domain taxonomy, domain-to-strategy table, strategy prompt strings, target model configuration, image ordering policy, answer parser, and Few-shot Complexity ranking rule.


## 2. Install dependencies

The notebook is designed for Kaggle or a standard Python environment with Internet access.

`google-genai==2.21.0` is pinned to match the project inference notebooks. The CLIP dependencies are used lazily and are only loaded if the Few-shot Complexity route is executed.

In [1]:
import importlib
import importlib.util
import importlib.metadata
import subprocess
import sys

# ------------------------------------------------------------------
# Version-aware dependency installation
# ------------------------------------------------------------------
# google-genai is pinned exactly because request/config semantics are part
# of the reproducibility contract. Other packages are installed only when
# missing.

GOOGLE_GENAI_REQUIRED_VERSION = "2.21.0"

REQUIREMENTS = {
    "datasets": ("datasets", "datasets>=3.0"),
    "huggingface_hub": ("huggingface-hub", "huggingface_hub>=0.24"),
    "PIL": ("Pillow", "Pillow>=10"),
    "pandas": ("pandas", "pandas>=2"),
    "numpy": ("numpy", "numpy>=1.26"),
    "pyarrow": ("pyarrow", "pyarrow>=15"),
    "tqdm": ("tqdm", "tqdm>=4.66"),
    "transformers": ("transformers", "transformers>=4.44,<5"),
    "torch": ("torch", "torch>=2.1"),
}


def installed_distribution_version(distribution_name):
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return None


install_specs = []

# Enforce the exact google-genai version even when an older Kaggle version
# is already installed.
installed_google_genai = installed_distribution_version("google-genai")

if installed_google_genai != GOOGLE_GENAI_REQUIRED_VERSION:
    install_specs.append(
        f"google-genai=={GOOGLE_GENAI_REQUIRED_VERSION}"
    )

# Install other packages only if their importable module is absent.
for module_name, (_distribution_name, pip_spec) in REQUIREMENTS.items():
    if importlib.util.find_spec(module_name) is None:
        install_specs.append(pip_spec)

if install_specs:
    print("Installing/upgrading dependencies:")
    for spec in install_specs:
        print("  -", spec)

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            *install_specs,
        ]
    )

    importlib.invalidate_caches()
else:
    print("All required dependencies already satisfy the frozen environment.")


# Verify the pinned SDK after installation.
verified_google_genai = installed_distribution_version("google-genai")

if verified_google_genai != GOOGLE_GENAI_REQUIRED_VERSION:
    raise RuntimeError(
        "google-genai installation verification failed: "
        f"observed={verified_google_genai!r}, "
        f"required={GOOGLE_GENAI_REQUIRED_VERSION!r}"
    )

# If google.genai had already been imported earlier in this same kernel
# with another version, its old in-memory code may survive a pip upgrade.
# A fresh 'Run All' starts clean and avoids this case.
if "google.genai" in sys.modules:
    raise RuntimeError(
        "google.genai was already imported before the pinned SDK installation. "
        "Restart the Kaggle session/kernel and Run All from the top so that "
        f"google-genai=={GOOGLE_GENAI_REQUIRED_VERSION} is loaded cleanly."
    )

print(
    "Dependency preflight PASS | "
    f"google-genai=={verified_google_genai}"
)

Installing/upgrading dependencies:
  - google-genai==2.21.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 59.3 MB/s eta 0:00:00
Dependency preflight PASS | google-genai==2.21.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.58.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.21.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.21.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2

## 3. Imports and immutable configuration

In [2]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter
from typing import Any, Iterable, Optional, Protocol
import ast
import base64
import hashlib
import importlib.metadata
import io
import json
import math
import os
import random
import re
import string
import time
import unicodedata

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download

from google import genai
from google.genai import types


# ------------------------------------------------------------------
# Target model / decoding
# ------------------------------------------------------------------

MODEL = "gemma-4-26b-a4b-it"
API_VERSION = "v1beta"
SDK_VERSION_PIN = GOOGLE_GENAI_REQUIRED_VERSION

TEMPERATURE = 0.0
TOP_P = None
TOP_K = None
API_SEED = 42
MAX_OUTPUT_TOKENS = 8192
CLASSIFIER_MAX_OUTPUT_TOKENS = 512
STREAM = False
THINKING_LEVEL = "minimal"

REQUEST_TIMEOUT_S = 1800
DEFERRED_REQUEST_TIMEOUT_S = 7200
MAX_ATTEMPTS = 3
DEFERRED_MAX_ATTEMPTS = 3
SDK_INTERNAL_RETRY_ATTEMPTS = 1
BASE_BACKOFF_S = 15.0
MAX_BACKOFF_S = 90.0
INTER_REQUEST_DELAY_S = 5.0
RETRYABLE_STATUS = {408, 409, 425, 429, 500, 502, 503, 504}

INLINE_REQUEST_SAFETY_BYTES = int(19.0 * 1024 * 1024)
FILE_UPLOAD_MAX_ATTEMPTS = 3
FILE_UPLOAD_PROCESSING_TIMEOUT_S = 180
FILE_UPLOAD_POLL_INTERVAL_S = 2.0

# ------------------------------------------------------------------
# Frozen Plan-and-Solve (P3) protocol — copied from the original
# Gemma 4 26B P3 notebook.
# ------------------------------------------------------------------
P3_PLAN_MAX_OUTPUT_TOKENS = 8192
P3_ANSWER_MAX_OUTPUT_TOKENS = 8192

P3_REQUEST_TIMEOUT_S = 1800
P3_DEFERRED_REQUEST_TIMEOUT_S = 7200
P3_MAX_ATTEMPTS = 3
P3_DEFERRED_MAX_ATTEMPTS = 3
P3_SDK_INTERNAL_RETRY_ATTEMPTS = 1
P3_BASE_BACKOFF_S = 15.0
P3_MAX_BACKOFF_S = 90.0
P3_INTER_REQUEST_DELAY_S = 5.0
P3_RETRYABLE_STATUS = {408, 409, 429, 500, 502, 503, 504}
P3_DEFERRED_RETRY_PASSES = 1
P3_MAX_CONSECUTIVE_EXHAUSTED_429 = 2
P3_SOFT_WALLCLOCK_HOURS = 10.5

P3_INLINE_REQUEST_LIMIT_BYTES = 20 * 1024 * 1024
P3_INLINE_REQUEST_SAFETY_BYTES = int(19.0 * 1024 * 1024)

P3_SCHEDULING_VERSION = (
    "two_stage_per_request_pacing_defer_timeout_then_long_retry_v1"
)

assert P3_PLAN_MAX_OUTPUT_TOKENS == P3_ANSWER_MAX_OUTPUT_TOKENS == 8192
assert P3_REQUEST_TIMEOUT_S == 1800
assert P3_DEFERRED_REQUEST_TIMEOUT_S == 7200
assert P3_MAX_ATTEMPTS == P3_DEFERRED_MAX_ATTEMPTS == 3
assert P3_DEFERRED_RETRY_PASSES == 1
assert P3_INTER_REQUEST_DELAY_S == 5.0
assert P3_MAX_CONSECUTIVE_EXHAUSTED_429 == 2

# ------------------------------------------------------------------
# Work directories
# ------------------------------------------------------------------

WORKDIR = Path("/kaggle/working/adaptive_domain_prompt_router")
MEDIA_DIR = WORKDIR / "media"
CACHE_DIR = WORKDIR / "cache"
DEMO_CACHE_DIR = CACHE_DIR / "mmmu_demo_source"
TRACE_PATH = WORKDIR / "inference_trace.jsonl"
API_EVENTS_PATH = WORKDIR / "api_events.jsonl"

# P3 plan and scheduler state are intentionally separate from the final
# benchmark JSONL so Call 1 and Call 2 can resume independently.
P3_PLAN_CHECKPOINT_PATH = WORKDIR / "plan_and_solve_plan_results.jsonl"
P3_DEFERRED_QUEUE_PATH = WORKDIR / "plan_and_solve_deferred_queue.json"

for path in (WORKDIR, MEDIA_DIR, CACHE_DIR, DEMO_CACHE_DIR):
    path.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Runtime policy
# ------------------------------------------------------------------

RUNTIME_CONFIG = {
    "prefer_valid_record_context": True,
    "classify_when_context_missing": True,
    "classifier_uses_images": True,
    "strict_domain_labels": True,
    "default_strategy_on_unrecoverable_route": "cot",
}

# ------------------------------------------------------------------
# Frozen routing policy
# ------------------------------------------------------------------

DOMAIN_ROUTER = {
    "Art and Design": "cot",
    "Business": "cot",
    "Health and Medicine": "fewshot_complexity",
    "Humanities and Social Science": "plan_and_solve",
    "Science": "ccot",
    "Tech and Engineering": "plan_and_solve",
}

DOMAIN_TO_SUBJECTS = {
    "Art and Design": [
        "Art", "Art_Theory", "Design", "Music",
    ],
    "Business": [
        "Accounting", "Economics", "Finance", "Manage", "Marketing",
    ],
    "Health and Medicine": [
        "Basic_Medical_Science", "Clinical_Medicine",
        "Diagnostics_and_Laboratory_Medicine", "Pharmacy", "Public_Health",
    ],
    "Humanities and Social Science": [
        "History", "Literature", "Sociology", "Psychology",
    ],
    "Science": [
        "Biology", "Chemistry", "Geography", "Math", "Physics",
    ],
    "Tech and Engineering": [
        "Agriculture", "Architecture_and_Engineering", "Computer_Science",
        "Electronics", "Energy_and_Power", "Materials",
        "Mechanical_Engineering",
    ],
}

SUBJECT_TO_DOMAIN = {
    subject: domain
    for domain, subjects in DOMAIN_TO_SUBJECTS.items()
    for subject in subjects
}

VALID_DOMAINS = tuple(DOMAIN_ROUTER)
VALID_SUBJECTS = tuple(SUBJECT_TO_DOMAIN)
VALID_DOMAIN_MODES = {"mmmu_pro", "api_classifier", "trained_classifier"}
ALL_LETTERS = list(string.ascii_uppercase)

assert set(DOMAIN_ROUTER) == set(DOMAIN_TO_SUBJECTS)
assert len(VALID_DOMAINS) == 6
assert len(VALID_SUBJECTS) == 30
assert set(DOMAIN_ROUTER.values()) == {
    "cot", "fewshot_complexity", "plan_and_solve", "ccot"
}

random.seed(API_SEED)
np.random.seed(API_SEED)

print("Model:", MODEL)
print("Domains:", len(VALID_DOMAINS))
print("Subjects:", len(VALID_SUBJECTS))
print("Router:", DOMAIN_ROUTER)

print(
    "P3 scheduler:",
    {
        "version": P3_SCHEDULING_VERSION,
        "main_timeout_s": P3_REQUEST_TIMEOUT_S,
        "deferred_timeout_s": P3_DEFERRED_REQUEST_TIMEOUT_S,
        "attempts_per_pass": P3_MAX_ATTEMPTS,
        "deferred_passes": P3_DEFERRED_RETRY_PASSES,
        "inter_request_delay_s": P3_INTER_REQUEST_DELAY_S,
        "plan_max_tokens": P3_PLAN_MAX_OUTPUT_TOKENS,
        "answer_max_tokens": P3_ANSWER_MAX_OUTPUT_TOKENS,
    },
)


Model: gemma-4-26b-a4b-it
Domains: 6
Subjects: 30
Router: {'Art and Design': 'cot', 'Business': 'cot', 'Health and Medicine': 'fewshot_complexity', 'Humanities and Social Science': 'plan_and_solve', 'Science': 'ccot', 'Tech and Engineering': 'plan_and_solve'}
P3 scheduler: {'version': 'two_stage_per_request_pacing_defer_timeout_then_long_retry_v1', 'main_timeout_s': 1800, 'deferred_timeout_s': 7200, 'attempts_per_pass': 3, 'deferred_passes': 1, 'inter_request_delay_s': 5.0, 'plan_max_tokens': 8192, 'answer_max_tokens': 8192}


## 4. Frozen prompt templates

The strings below are copied from the corresponding project experiments and guarded by SHA-256 checks. Any accidental modification causes an immediate failure before inference.

In [3]:
# ------------------------------------------------------------------
# CoT
# ------------------------------------------------------------------

COT_SUFFIX = """Answer the preceding multiple choice question.
The last line of your response should be of the following format:
'Answer: $LETTER' (without quotes) where LETTER is one of options.
Think step by step before answering.
Explain your reasoning before answering."""

EXPECTED_COT_SUFFIX_SHA256 = (
    "a17f20268b2d36bfbcaca5538f20db0a6991e4c0d149255c5aa26f645f2cb9a0"
)

# ------------------------------------------------------------------
# CCoT / per-image scene graph
# ------------------------------------------------------------------

SCENE_GRAPH_INSTRUCTION = (
    "For the provided image and its associated question, generate a scene graph "
    "in JSON format that includes the following:\n\n"
    "1. Objects that are relevant to answering the question.\n"
    "2. Object attributes that are relevant to answering the question.\n"
    "3. Object relationships that are relevant to answering the question.\n\n"
    "Scene Graph:"
)

CCOT_FINAL_SUFFIX = """Answer the preceding multiple choice question.
The last line of your response should be of the following format:
'Answer: $LETTER' (without quotes) where LETTER is one of options.
Explain your reasoning before answering."""

EXPECTED_SCENE_GRAPH_INSTRUCTION_SHA256 = (
    "3a37bf27bd67ecb3437e7d3666c9ee876a31f8bea8a8e07874ae64d759a0c560"
)
EXPECTED_CCOT_FINAL_SUFFIX_SHA256 = (
    "acac7da8792f4b0fef57eaf1c1a2ac3668dc63ac503f936f3dea1b3e2e8a91d0"
)

# ------------------------------------------------------------------
# Plan-and-Solve
# ------------------------------------------------------------------

PLAN_SUFFIX = (
    "A: Let's first understand the problem and devise a plan to solve the problem."
)

PLAN_EXECUTION_SUFFIX = """Answer the preceding multiple choice question.
The last line of your response should be of the following format:
'Answer: $LETTER' (without quotes) where LETTER is one of options.
let's carry out the plan and solve the problem step by step.
Explain your reasoning before answering."""

EXPECTED_PLAN_SUFFIX_SHA256 = (
    "766a5e22674ee450f6365fa5091eb9a4209d3e97712c8c5f3a61502a5885ffb6"
)
EXPECTED_PLAN_EXECUTION_SUFFIX_SHA256 = (
    "e59c1b1b8f910d2f45db6bdf00207b59fb8b77cc7c9b4c934ef239bb4afcbbe2"
)

# ------------------------------------------------------------------
# Few-shot Complexity
# ------------------------------------------------------------------

FEWSHOT_INSTRUCTION = """Carefully review the given image and the associated text. Utilize the reasoning format
illustrated in the provided examples, breaking down your thought process. Ensure that each
reasoning step is explicitly connected to observable details in the image or text, and articulate
your conclusion in a clear and logical manner.

The number of answer choices may vary across demonstrations.
For the final multiple-choice question, select exactly one of the option labels provided
with that question.

Explain your reasoning before answering.
The last line of your response should be of the following format:
'Answer: $LETTER' (without quotes), where LETTER is one of the option labels
provided in the final question."""

EXPECTED_FEWSHOT_INSTRUCTION_SHA256 = (
    "81b1b7250e505e4219bfa5b226d501094e6e4ce364d7a25eb6bb9c025ba09fbc"
)

# ------------------------------------------------------------------
# Hash gate
# ------------------------------------------------------------------

def text_sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

assert text_sha256(COT_SUFFIX) == EXPECTED_COT_SUFFIX_SHA256
assert (
    text_sha256(SCENE_GRAPH_INSTRUCTION)
    == EXPECTED_SCENE_GRAPH_INSTRUCTION_SHA256
)
assert (
    text_sha256(CCOT_FINAL_SUFFIX)
    == EXPECTED_CCOT_FINAL_SUFFIX_SHA256
)
assert text_sha256(PLAN_SUFFIX) == EXPECTED_PLAN_SUFFIX_SHA256
assert (
    text_sha256(PLAN_EXECUTION_SUFFIX)
    == EXPECTED_PLAN_EXECUTION_SUFFIX_SHA256
)
assert (
    text_sha256(FEWSHOT_INSTRUCTION)
    == EXPECTED_FEWSHOT_INSTRUCTION_SHA256
)

PROMPT_HASHES = {
    "cot_suffix": text_sha256(COT_SUFFIX),
    "scene_graph_instruction": text_sha256(SCENE_GRAPH_INSTRUCTION),
    "ccot_final_suffix": text_sha256(CCOT_FINAL_SUFFIX),
    "plan_suffix": text_sha256(PLAN_SUFFIX),
    "plan_execution_suffix": text_sha256(PLAN_EXECUTION_SUFFIX),
    "fewshot_instruction": text_sha256(FEWSHOT_INSTRUCTION),
}

print(json.dumps(PROMPT_HASHES, indent=2))

{
  "cot_suffix": "a17f20268b2d36bfbcaca5538f20db0a6991e4c0d149255c5aa26f645f2cb9a0",
  "scene_graph_instruction": "3a37bf27bd67ecb3437e7d3666c9ee876a31f8bea8a8e07874ae64d759a0c560",
  "ccot_final_suffix": "acac7da8792f4b0fef57eaf1c1a2ac3668dc63ac503f936f3dea1b3e2e8a91d0",
  "plan_suffix": "766a5e22674ee450f6365fa5091eb9a4209d3e97712c8c5f3a61502a5885ffb6",
  "plan_execution_suffix": "e59c1b1b8f910d2f45db6bdf00207b59fb8b77cc7c9b4c934ef239bb4afcbbe2",
  "fewshot_instruction": "81b1b7250e505e4219bfa5b226d501094e6e4ce364d7a25eb6bb9c025ba09fbc"
}


## 5. Input representation and option parsing

A user can pass either:

```python
question="... \nA. choice one\nB. choice two\nC. choice three"
```

or:

```python
question="..."
options=["choice one", "choice two", "choice three"]
```

Images may be supplied as `PIL.Image`, local file paths, raw bytes, or byte-bearing dictionaries.

If numbered image markers such as `<image 1>` are present, the marker occurrences define image order. Repeated references remain repeated. If no numbered markers are present, the supplied image-list order is used.

In [4]:
IMAGE_MARKER_RE = re.compile(r"<image\s+(\d+)>", flags=re.IGNORECASE)
OPTION_HEADER_RE = re.compile(
    r"(?m)^[ \t]*([A-Z])[\.\)][ \t]+"
)


def choice_letters(n: int) -> list[str]:
    n = int(n)
    if not 2 <= n <= len(ALL_LETTERS):
        raise ValueError(f"Unsupported number of choices: {n}")
    return ALL_LETTERS[:n]


def parse_options_value(value: Any) -> list[str]:
    if isinstance(value, (list, tuple)):
        options = list(value)
    elif hasattr(value, "tolist") and not isinstance(value, str):
        options = list(value.tolist())
    elif isinstance(value, str):
        options = None
        for parser in (ast.literal_eval, json.loads):
            try:
                candidate = parser(value.strip())
                if isinstance(candidate, (list, tuple)):
                    options = list(candidate)
                    break
            except Exception:
                pass
        if options is None:
            raise ValueError("Could not parse serialized options.")
    else:
        raise TypeError(f"Unsupported options type: {type(value)}")

    options = [str(x).strip() for x in options]
    choice_letters(len(options))
    return options


def split_embedded_options(text: str) -> tuple[str, list[str]]:
    text = str(text).strip()
    matches = list(OPTION_HEADER_RE.finditer(text))

    if len(matches) < 2:
        raise ValueError(
            "No explicit options were supplied and sequential option lines "
            "could not be recovered from the question. Include choices as "
            "'A. ...', 'B. ...', ... or pass options=[...]."
        )

    labels = [match.group(1).upper() for match in matches]
    expected = choice_letters(len(labels))
    if labels != expected:
        raise ValueError(
            f"Embedded choices must be sequential from A. Observed={labels}, "
            f"expected={expected}."
        )

    stem = text[:matches[0].start()].strip()
    options = []

    for i, match in enumerate(matches):
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        options.append(text[start:end].strip())

    if not stem:
        raise ValueError("Question stem is empty after option extraction.")
    if any(not option for option in options):
        raise ValueError("At least one parsed option is empty.")

    return stem, options


def normalize_marker_text(text: str) -> str:
    return IMAGE_MARKER_RE.sub("<image>", str(text))


def safe_filename(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(text))[:120]


def image_to_pil(value: Any, source_dir: Optional[Path] = None) -> Image.Image:
    if isinstance(value, Image.Image):
        image = value.copy()
        image.load()
        return image

    payload = None

    if isinstance(value, memoryview):
        payload = value.tobytes()
    elif isinstance(value, (bytes, bytearray)):
        payload = bytes(value)
    elif isinstance(value, dict):
        raw = value.get("bytes")
        if isinstance(raw, memoryview):
            raw = raw.tobytes()
        if isinstance(raw, (bytes, bytearray)):
            payload = bytes(raw)
        elif value.get("path"):
            value = value["path"]

    if payload is None and isinstance(value, (str, Path)):
        path_candidates = [Path(value)]
        if source_dir is not None:
            path_candidates.append(Path(source_dir) / str(value))
        for candidate in path_candidates:
            if candidate.is_file():
                payload = candidate.read_bytes()
                break

    if payload is None:
        raise TypeError(
            "Image must be a PIL image, local path, bytes, memoryview, "
            "or a dict containing bytes/path."
        )

    with Image.open(io.BytesIO(payload)) as opened:
        opened.load()
        return opened.copy()


def lossless_png_bytes(image: Image.Image) -> bytes:
    image = image.copy()
    if image.mode == "P" and "transparency" in image.info:
        image = image.convert("RGBA")
    elif image.mode not in ("RGB", "RGBA", "L", "LA", "P"):
        image = image.convert("RGB")

    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return buffer.getvalue()


@dataclass(frozen=True)
class ImageAsset:
    index: int
    png_bytes: bytes
    sha256: str
    path: str
    rices_path: str
    width: int
    height: int

    @classmethod
    def from_input(
        cls,
        index: int,
        value: Any,
        media_dir: Path = MEDIA_DIR,
        source_dir: Optional[Path] = None,
    ) -> "ImageAsset":
        image = image_to_pil(value, source_dir=source_dir)

        # Prompt transport remains unchanged: archive the image exactly through
        # the pre-existing lossless-PNG path.
        data = lossless_png_bytes(image)
        digest = hashlib.sha256(data).hexdigest()

        path = media_dir / f"image_{index:02d}_{digest[:16]}.png"
        if not path.exists():
            path.write_bytes(data)
        elif hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Image cache checksum mismatch: {path}")

        # RICES preprocessing is isolated from prompt transport and mirrors the
        # frozen embedding notebook: EXIF orientation -> RGB -> CLIPProcessor.
        rices_image = ImageOps.exif_transpose(image.copy()).convert("RGB")
        rices_data = lossless_png_bytes(rices_image)
        rices_digest = hashlib.sha256(rices_data).hexdigest()
        rices_path = media_dir / (
            f"rices_image_{index:02d}_{rices_digest[:16]}.png"
        )
        if not rices_path.exists():
            rices_path.write_bytes(rices_data)
        elif hashlib.sha256(rices_path.read_bytes()).hexdigest() != rices_digest:
            raise RuntimeError(
                f"RICES image cache checksum mismatch: {rices_path}"
            )

        return cls(
            index=index,
            png_bytes=data,
            sha256=digest,
            path=str(path),
            rices_path=str(rices_path),
            width=int(image.width),
            height=int(image.height),
        )


@dataclass
class ImageOccurrence:
    occurrence_index: int
    image_ref: int
    source_section: str
    option_letter: Optional[str]
    image_label: str
    context_placeholder: str
    graph_heading: str


@dataclass
class Problem:
    question: str
    options: list[str]
    images: list[ImageAsset]
    sample_id: str = "interactive"
    subject: Optional[str] = None
    domain: Optional[str] = None
    difficulty: Optional[str] = None
    occurrences: list[ImageOccurrence] = field(default_factory=list)
    markers_present: bool = False

    @property
    def letters(self) -> list[str]:
        return choice_letters(len(self.options))

    @property
    def image_map(self) -> dict[int, ImageAsset]:
        return {image.index: image for image in self.images}


def build_problem(
    question: str,
    images: Iterable[Any],
    options: Optional[Iterable[str]] = None,
    sample_id: str = "interactive",
    subject: Optional[str] = None,
    domain: Optional[str] = None,
    difficulty: Optional[str] = None,
    source_dir: Optional[Path] = None,
) -> Problem:
    if options is None:
        question_stem, parsed_options = split_embedded_options(question)
    else:
        question_stem = str(question).strip()
        parsed_options = [str(x).strip() for x in options]
        choice_letters(len(parsed_options))

    image_values = list(images)
    if not image_values:
        raise ValueError("At least one image is required.")

    image_assets = [
        ImageAsset.from_input(
            i + 1,
            value,
            source_dir=source_dir,
        )
        for i, value in enumerate(image_values)
    ]

    problem = Problem(
        question=question_stem,
        options=parsed_options,
        images=image_assets,
        sample_id=str(sample_id),
        subject=subject,
        domain=domain,
        difficulty=difficulty,
    )

    problem.occurrences, problem.markers_present = analyze_occurrences(problem)
    return problem


def analyze_occurrences(
    problem: Problem,
) -> tuple[list[ImageOccurrence], bool]:
    segments = [("question", None, problem.question)]
    segments.extend(
        ("option", letter, option)
        for letter, option in zip(problem.letters, problem.options)
    )

    refs_by_segment = {
        (kind, letter): [
            int(match.group(1))
            for match in IMAGE_MARKER_RE.finditer(text)
        ]
        for kind, letter, text in segments
    }

    all_refs = [
        ref
        for refs in refs_by_segment.values()
        for ref in refs
    ]

    available = set(range(1, len(problem.images) + 1))

    if not all_refs:
        occurrences = []
        for image in problem.images:
            ordinal = image.index
            occurrences.append(
                ImageOccurrence(
                    occurrence_index=len(occurrences),
                    image_ref=image.index,
                    source_section="question",
                    option_letter=None,
                    image_label=f"Question Image {ordinal}",
                    context_placeholder=f"[Question Image {ordinal}]",
                    graph_heading=f"Scene Graph for Question Image {ordinal}:",
                )
            )
        return occurrences, False

    if set(all_refs) != available:
        raise ValueError(
            "Numbered image markers must reference every supplied image exactly "
            "within the available 1..N index set. "
            f"Observed refs={sorted(set(all_refs))}; available={sorted(available)}."
        )

    question_unique = list(
        dict.fromkeys(refs_by_segment[("question", None)])
    )
    question_index = {
        ref: i + 1
        for i, ref in enumerate(question_unique)
    }

    option_unique = {
        letter: list(
            dict.fromkeys(refs_by_segment[("option", letter)])
        )
        for letter in problem.letters
    }

    occurrences = []

    for kind, option_letter, _text in segments:
        refs = refs_by_segment[(kind, option_letter)]

        for ref in refs:
            if kind == "question":
                ordinal = question_index[ref]
                image_label = f"Question Image {ordinal}"
                context_placeholder = f"[Question Image {ordinal}]"
                graph_heading = f"Scene Graph for Question Image {ordinal}:"
            else:
                uniques = option_unique[option_letter]
                ordinal = uniques.index(ref) + 1

                if len(uniques) == 1:
                    image_label = f"Option {option_letter} Image"
                    context_placeholder = f"[visual option {option_letter}]"
                    graph_heading = f"Scene Graph for Option {option_letter}:"
                else:
                    image_label = (
                        f"Option {option_letter} Image {ordinal}"
                    )
                    context_placeholder = (
                        f"[visual option {option_letter} image {ordinal}]"
                    )
                    graph_heading = (
                        f"Scene Graph for Option {option_letter} "
                        f"Image {ordinal}:"
                    )

            occurrences.append(
                ImageOccurrence(
                    occurrence_index=len(occurrences),
                    image_ref=ref,
                    source_section=kind,
                    option_letter=option_letter,
                    image_label=image_label,
                    context_placeholder=context_placeholder,
                    graph_heading=graph_heading,
                )
            )

    return occurrences, True


def format_options(options: Iterable[str]) -> str:
    options = list(options)
    return "\n".join(
        f"{letter}. {option}"
        for letter, option in zip(choice_letters(len(options)), options)
    )


def physical_images_in_occurrence_order(problem: Problem) -> list[ImageAsset]:
    image_map = problem.image_map
    return [
        image_map[occ.image_ref]
        for occ in problem.occurrences
    ]


print("Input utilities loaded.")

Input utilities loaded.


## 6. Prompt-part primitives and semantic rendering

In [5]:
@dataclass
class PromptPart:
    kind: str
    text: Optional[str] = None
    image: Optional[ImageAsset] = None
    label: Optional[str] = None

    @staticmethod
    def text_part(text: str) -> "PromptPart":
        return PromptPart(kind="text", text=str(text))

    @staticmethod
    def image_part(
        image: ImageAsset,
        label: Optional[str] = None,
    ) -> "PromptPart":
        return PromptPart(kind="image", image=image, label=label)


def parts_sha256(parts: list[PromptPart]) -> str:
    canonical = []
    for part in parts:
        if part.kind == "text":
            canonical.append({"type": "text", "text": part.text})
        elif part.kind == "image":
            canonical.append({
                "type": "image",
                "sha256": part.image.sha256,
                "label": part.label,
            })
        else:
            raise RuntimeError(f"Unknown prompt part kind: {part.kind}")
    payload = json.dumps(
        canonical,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def occurrences_for_segment(
    problem: Problem,
    section: str,
    option_letter: Optional[str],
) -> list[ImageOccurrence]:
    return [
        occ
        for occ in problem.occurrences
        if occ.source_section == section
        and occ.option_letter == option_letter
    ]


def replace_segment_markers(
    text: str,
    occurrences: list[ImageOccurrence],
) -> str:
    if not occurrences:
        return str(text).strip()

    placeholder_by_ref = {
        occ.image_ref: occ.context_placeholder
        for occ in occurrences
    }

    return IMAGE_MARKER_RE.sub(
        lambda match: placeholder_by_ref[int(match.group(1))],
        str(text).strip(),
    )


def semantic_question_and_options(
    problem: Problem,
) -> tuple[str, str]:
    if not problem.markers_present:
        return problem.question.strip(), format_options(problem.options)

    q_occ = occurrences_for_segment(
        problem,
        "question",
        None,
    )
    question_text = replace_segment_markers(
        problem.question,
        q_occ,
    )

    rendered_options = []
    for letter, option in zip(problem.letters, problem.options):
        occ = occurrences_for_segment(
            problem,
            "option",
            letter,
        )
        rendered = replace_segment_markers(
            option,
            occ,
        )
        rendered_options.append(f"{letter}. {rendered}")

    return question_text, "\n".join(rendered_options)


def image_prefix_parts(
    problem: Problem,
    include_labels: bool,
) -> list[PromptPart]:
    parts = []
    image_map = problem.image_map

    for occ in problem.occurrences:
        if include_labels:
            parts.append(
                PromptPart.text_part(f"[{occ.image_label}]")
            )
        parts.append(
            PromptPart.image_part(
                image_map[occ.image_ref],
                label=occ.image_label,
            )
        )

    return parts


def append_interleaved_segment(
    parts: list[PromptPart],
    problem: Problem,
    text: str,
    occurrences: list[ImageOccurrence],
    prefix: str = "",
    suffix: str = "",
):
    if prefix:
        parts.append(PromptPart.text_part(prefix))

    if not problem.markers_present:
        parts.append(PromptPart.text_part(str(text) + suffix))
        return

    matches = list(IMAGE_MARKER_RE.finditer(str(text)))
    if len(matches) != len(occurrences):
        raise RuntimeError(
            "Image occurrence count does not match text markers."
        )

    cursor = 0
    image_map = problem.image_map

    for match, occ in zip(matches, occurrences):
        if match.start() > cursor:
            parts.append(
                PromptPart.text_part(str(text)[cursor:match.start()])
            )

        parts.append(
            PromptPart.text_part(f"[{occ.image_label}]\n")
        )
        parts.append(
            PromptPart.image_part(
                image_map[occ.image_ref],
                label=occ.image_label,
            )
        )
        parts.append(PromptPart.text_part("\n"))
        cursor = match.end()

    if cursor < len(str(text)):
        parts.append(
            PromptPart.text_part(str(text)[cursor:] + suffix)
        )
    elif suffix:
        parts.append(PromptPart.text_part(suffix))


def append_problem_interleaved(
    parts: list[PromptPart],
    problem: Problem,
):
    if not problem.markers_present:
        for occ in problem.occurrences:
            parts.append(
                PromptPart.text_part(f"[{occ.image_label}]\n")
            )
            parts.append(
                PromptPart.image_part(
                    problem.image_map[occ.image_ref],
                    label=occ.image_label,
                )
            )
            parts.append(PromptPart.text_part("\n"))

        parts.append(
            PromptPart.text_part(problem.question + "\n\n")
        )
        for letter, option in zip(problem.letters, problem.options):
            parts.append(
                PromptPart.text_part(
                    f"{letter}. {option}\n"
                )
            )
        return

    q_occ = occurrences_for_segment(
        problem,
        "question",
        None,
    )
    append_interleaved_segment(
        parts,
        problem,
        problem.question,
        q_occ,
        suffix="\n\n",
    )

    for letter, option in zip(problem.letters, problem.options):
        occ = occurrences_for_segment(
            problem,
            "option",
            letter,
        )
        append_interleaved_segment(
            parts,
            problem,
            option,
            occ,
            prefix=f"{letter}. ",
            suffix="\n",
        )


print("Prompt-part utilities loaded.")

Prompt-part utilities loaded.


## 7. Google Gemma API client

The API key can be passed directly to `AdaptivePromptRouter(api_key=...)`. If omitted, the notebook attempts to read `GEMINI_API_KEY` or `GOOGLE_API_KEY` from Kaggle Secrets and then from environment variables.

The key is never written to an output file or printed.

In [6]:
def resolve_api_key(api_key: Optional[str] = None) -> tuple[str, str]:
    if api_key:
        return str(api_key).strip(), "explicit_argument"

    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
            try:
                value = secrets.get_secret(name)
            except Exception:
                value = None

            if value:
                return value.strip(), f"kaggle_secret:{name}"
    except Exception:
        pass

    for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
        value = os.environ.get(name)
        if value:
            return value.strip(), f"environment:{name}"

    raise RuntimeError(
        "No Google API key was supplied. Pass api_key=... or configure "
        "GEMINI_API_KEY / GOOGLE_API_KEY."
    )


def append_jsonl(path: Path, row: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(
            json.dumps(
                row,
                ensure_ascii=False,
                default=str,
            )
            + "\n"
        )
        handle.flush()
        os.fsync(handle.fileno())


def obj_get(obj: Any, name: str, default=None):
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


def enum_text(value: Any) -> Optional[str]:
    if value is None:
        return None
    enum_value = getattr(value, "value", None)
    return str(enum_value) if enum_value is not None else str(value)


def status_from_exception(exc: Exception) -> Optional[int]:
    for name in ("status_code", "code"):
        value = getattr(exc, name, None)
        if value is not None:
            try:
                return int(value)
            except Exception:
                nested = getattr(value, "value", None)
                if nested is not None:
                    try:
                        return int(nested)
                    except Exception:
                        pass

    response = getattr(exc, "response", None)
    if response is not None:
        try:
            return int(getattr(response, "status_code"))
        except Exception:
            pass

    match = re.search(r"\b(4\d\d|5\d\d)\b", str(exc))
    return int(match.group(1)) if match else None


def retry_after_seconds(exc: Exception, fallback: float) -> float:
    response = getattr(exc, "response", None)
    headers = getattr(response, "headers", None) if response is not None else None

    if headers:
        try:
            value = (
                headers.get("retry-after")
                or headers.get("Retry-After")
            )
            if value is not None:
                return max(float(value), fallback)
        except Exception:
            pass

    for pattern in (
        r"retry(?:Delay| after)?[^0-9]{0,20}"
        r"([0-9]+(?:\.[0-9]+)?)\s*s",
        r"Please retry in ([0-9]+(?:\.[0-9]+)?)s",
    ):
        match = re.search(
            pattern,
            str(exc),
            flags=re.IGNORECASE,
        )
        if match:
            return max(float(match.group(1)), fallback)

    return fallback


class GoogleGemmaClient:
    def __init__(self, api_key: Optional[str] = None):
        self.api_key, self.api_key_source = resolve_api_key(api_key)

        installed_sdk = importlib.metadata.version("google-genai")
        if installed_sdk != SDK_VERSION_PIN:
            raise RuntimeError(
                f"google-genai version mismatch: "
                f"{installed_sdk} != {SDK_VERSION_PIN}. "
                f"Install google-genai=={SDK_VERSION_PIN}."
            )

        self.client = self._make_client(
            REQUEST_TIMEOUT_S * 1000
        )
        self.deferred_client = self._make_client(
            DEFERRED_REQUEST_TIMEOUT_S * 1000
        )

        self.last_request_started = None
        self.file_cache = {}
        self.call_counter = 0

    def _make_client(self, timeout_ms: int):
        return genai.Client(
            api_key=self.api_key,
            http_options=types.HttpOptions(
                api_version=API_VERSION,
                timeout=int(timeout_ms),
                retry_options=types.HttpRetryOptions(
                    attempts=SDK_INTERNAL_RETRY_ATTEMPTS,
                ),
            ),
        )

    def _generation_config(self, max_tokens: int):
        return types.GenerateContentConfig(
            temperature=TEMPERATURE,
            seed=API_SEED,
            max_output_tokens=int(max_tokens),
            thinking_config=types.ThinkingConfig(
                thinking_level=THINKING_LEVEL,
            ),
        )

    @staticmethod
    def _estimate_inline_bytes(parts: list[PromptPart]) -> int:
        text_bytes = sum(
            len((part.text or "").encode("utf-8"))
            for part in parts
            if part.kind == "text"
        )
        base64_bytes = sum(
            4 * math.ceil(len(part.image.png_bytes) / 3)
            for part in parts
            if part.kind == "image"
        )
        return int(
            text_bytes
            + base64_bytes
            + 4096
            + 256 * len(parts)
        )

    @staticmethod
    def _file_state(file_obj: Any) -> str:
        return enum_text(obj_get(file_obj, "state", "")).upper()

    def _upload_image(
        self,
        image: ImageAsset,
        active_client,
        event_base: dict,
    ):
        cached = self.file_cache.get(image.sha256)
        if cached is not None:
            return types.Part.from_uri(
                file_uri=cached["uri"],
                mime_type="image/png",
            )

        last_exc = None

        for attempt in range(1, FILE_UPLOAD_MAX_ATTEMPTS + 1):
            try:
                uploaded = active_client.files.upload(
                    file=image.path
                )

                deadline = (
                    time.monotonic()
                    + FILE_UPLOAD_PROCESSING_TIMEOUT_S
                )
                state = self._file_state(uploaded)

                while state.endswith("PROCESSING"):
                    if time.monotonic() >= deadline:
                        raise TimeoutError(
                            "Google Files API processing timeout."
                        )

                    time.sleep(FILE_UPLOAD_POLL_INTERVAL_S)
                    uploaded = active_client.files.get(
                        name=uploaded.name
                    )
                    state = self._file_state(uploaded)

                if state and not state.endswith("ACTIVE"):
                    raise RuntimeError(
                        f"Uploaded file did not become active: {state}"
                    )

                uri = str(obj_get(uploaded, "uri", "") or "")
                if not uri:
                    raise RuntimeError(
                        "Google Files API returned no URI."
                    )

                self.file_cache[image.sha256] = {
                    "uri": uri,
                    "name": str(obj_get(uploaded, "name", "") or ""),
                }

                append_jsonl(
                    API_EVENTS_PATH,
                    {
                        **event_base,
                        "event": "file_upload_success",
                        "image_sha256": image.sha256,
                        "attempt": attempt,
                        "utc": datetime.now(timezone.utc).isoformat(),
                    },
                )

                return types.Part.from_uri(
                    file_uri=uri,
                    mime_type="image/png",
                )

            except Exception as exc:
                last_exc = exc
                status = status_from_exception(exc)
                retryable = (
                    status is None
                    or status in RETRYABLE_STATUS
                )
                will_retry = (
                    retryable
                    and attempt < FILE_UPLOAD_MAX_ATTEMPTS
                )

                append_jsonl(
                    API_EVENTS_PATH,
                    {
                        **event_base,
                        "event": "file_upload_error",
                        "image_sha256": image.sha256,
                        "attempt": attempt,
                        "http_status": status,
                        "will_retry": will_retry,
                        "error_type": type(exc).__name__,
                        "error_message": str(exc)[:4000],
                        "utc": datetime.now(timezone.utc).isoformat(),
                    },
                )

                if not will_retry:
                    raise

                fallback = min(
                    MAX_BACKOFF_S,
                    BASE_BACKOFF_S * (2 ** (attempt - 1)),
                )
                time.sleep(
                    min(
                        MAX_BACKOFF_S,
                        retry_after_seconds(exc, fallback),
                    )
                )

        raise last_exc or RuntimeError("Image upload failed.")

    def _to_google_contents(
        self,
        parts: list[PromptPart],
        active_client,
        transport_mode: str,
        event_base: dict,
    ):
        google_parts = []

        for part in parts:
            if part.kind == "text":
                google_parts.append(
                    types.Part.from_text(text=part.text or "")
                )
                continue

            if part.kind != "image" or part.image is None:
                raise RuntimeError(
                    f"Invalid prompt part: {part}"
                )

            if transport_mode == "inline_png":
                google_parts.append(
                    types.Part.from_bytes(
                        data=part.image.png_bytes,
                        mime_type="image/png",
                    )
                )
            elif transport_mode == "google_files_uri_png":
                google_parts.append(
                    self._upload_image(
                        part.image,
                        active_client,
                        event_base,
                    )
                )
            else:
                raise RuntimeError(
                    f"Unknown transport mode: {transport_mode}"
                )

        return types.UserContent(parts=google_parts)

    @staticmethod
    def _extract_response(response) -> dict:
        candidates = obj_get(response, "candidates", None) or []
        first = candidates[0] if candidates else None
        content = obj_get(first, "content", None)

        visible_parts = []
        thought_parts = []

        for part in obj_get(content, "parts", None) or []:
            text = obj_get(part, "text", None)
            if text is None:
                continue

            if bool(obj_get(part, "thought", False)):
                thought_parts.append(str(text))
            else:
                visible_parts.append(str(text))

        visible = "".join(visible_parts).strip()

        if not visible:
            try:
                visible = str(response.text or "").strip()
            except Exception:
                visible = ""

        reasoning = "\n".join(thought_parts).strip()

        usage = obj_get(response, "usage_metadata", None)
        completion_tokens = obj_get(
            usage,
            "candidates_token_count",
            None,
        )
        if completion_tokens is None:
            completion_tokens = obj_get(
                usage,
                "response_token_count",
                None,
            )

        return {
            "text": visible,
            "provider_reasoning": reasoning,
            "prompt_tokens": obj_get(
                usage,
                "prompt_token_count",
                None,
            ),
            "completion_tokens": completion_tokens,
            "reasoning_tokens": obj_get(
                usage,
                "thoughts_token_count",
                None,
            ),
            "total_tokens": obj_get(
                usage,
                "total_token_count",
                None,
            ),
            "finish_reason": enum_text(
                obj_get(first, "finish_reason", None)
            ),
            "response_id": obj_get(
                response,
                "response_id",
                None,
            ),
            "model_version": obj_get(
                response,
                "model_version",
                None,
            ),
        }

    def _pace(self):
        if self.last_request_started is None:
            return

        elapsed = time.monotonic() - self.last_request_started
        remaining = INTER_REQUEST_DELAY_S - elapsed

        if remaining > 0:
            time.sleep(remaining)

    def _send_pass(
        self,
        parts: list[PromptPart],
        purpose: str,
        sample_id: str,
        max_tokens: int,
        active_client,
        timeout_s: int,
        attempt_limit: int,
        pass_name: str,
    ) -> dict:
        inline_estimate = self._estimate_inline_bytes(parts)
        transport_mode = (
            "inline_png"
            if inline_estimate < INLINE_REQUEST_SAFETY_BYTES
            else "google_files_uri_png"
        )

        prompt_hash = parts_sha256(parts)
        last_exc = None

        for attempt in range(1, attempt_limit + 1):
            self._pace()
            self.last_request_started = time.monotonic()
            self.call_counter += 1

            started = time.perf_counter()
            event_base = {
                "sample_id": sample_id,
                "purpose": purpose,
                "pass": pass_name,
                "attempt": attempt,
                "model": MODEL,
                "temperature": TEMPERATURE,
                "seed": API_SEED,
                "max_output_tokens": int(max_tokens),
                "thinking_level": THINKING_LEVEL,
                "top_p_sent": False,
                "top_k_sent": False,
                "prompt_sha256": prompt_hash,
                "transport_mode": transport_mode,
                "inline_payload_estimate_bytes": inline_estimate,
            }

            print(
                f"[API START] sample={sample_id} | purpose={purpose} | "
                f"pass={pass_name} | attempt={attempt}/{attempt_limit} | "
                f"call_counter={self.call_counter} | timeout_s={timeout_s} | "
                f"transport={transport_mode}",
                flush=True,
            )

            try:
                contents = self._to_google_contents(
                    parts,
                    active_client,
                    transport_mode,
                    event_base,
                )

                response = active_client.models.generate_content(
                    model=MODEL,
                    contents=contents,
                    config=self._generation_config(max_tokens),
                )

                elapsed = time.perf_counter() - started
                parsed = self._extract_response(response)

                reasoning_tokens = parsed["reasoning_tokens"]
                reasoning_violation = (
                    bool(parsed["provider_reasoning"].strip())
                    or (
                        reasoning_tokens is not None
                        and int(reasoning_tokens) > 0
                    )
                    or bool(
                        re.search(
                            r"</?think(?:\s[^>]*)?>",
                            parsed["text"],
                            flags=re.IGNORECASE,
                        )
                    )
                )

                if reasoning_violation:
                    raise RuntimeError(
                        "Provider-native reasoning evidence appeared "
                        "despite thinking_level='minimal'."
                    )

                result = {
                    "api_ok": True,
                    "text": parsed["text"],
                    "latency_s": elapsed,
                    "attempt": attempt,
                    "pass": pass_name,
                    "transport_mode": transport_mode,
                    "prompt_sha256": prompt_hash,
                    "prompt_tokens": parsed["prompt_tokens"],
                    "completion_tokens": parsed["completion_tokens"],
                    "reasoning_tokens": parsed["reasoning_tokens"],
                    "total_tokens": parsed["total_tokens"],
                    "finish_reason": parsed["finish_reason"],
                    "response_id": parsed["response_id"],
                    "model_version": parsed["model_version"],
                    "purpose": purpose,
                }

                append_jsonl(
                    API_EVENTS_PATH,
                    {
                        **event_base,
                        **result,
                        "event": "request_success",
                        "utc": datetime.now(timezone.utc).isoformat(),
                    },
                )

                print(
                    f"[API OK] sample={sample_id} | purpose={purpose} | "
                    f"pass={pass_name} | attempt={attempt}/{attempt_limit} | "
                    f"elapsed_s={elapsed:.1f} | "
                    f"prompt_tokens={parsed['prompt_tokens']} | "
                    f"completion_tokens={parsed['completion_tokens']}",
                    flush=True,
                )

                return result

            except Exception as exc:
                last_exc = exc
                elapsed = time.perf_counter() - started
                status = status_from_exception(exc)
                retryable = (
                    status is None
                    or status in RETRYABLE_STATUS
                    or isinstance(exc, TimeoutError)
                )
                will_retry = (
                    retryable
                    and attempt < attempt_limit
                )

                append_jsonl(
                    API_EVENTS_PATH,
                    {
                        **event_base,
                        "event": "request_error",
                        "latency_s": elapsed,
                        "http_status": status,
                        "will_retry": will_retry,
                        "error_type": type(exc).__name__,
                        "error_message": str(exc)[:8000],
                        "utc": datetime.now(timezone.utc).isoformat(),
                    },
                )

                print(
                    f"[API ERROR] sample={sample_id} | purpose={purpose} | "
                    f"pass={pass_name} | attempt={attempt}/{attempt_limit} | "
                    f"elapsed_s={elapsed:.1f} | status={status} | "
                    f"retry={will_retry} | {type(exc).__name__}: {str(exc)[:300]}",
                    flush=True,
                )

                if not will_retry:
                    break

                fallback = min(
                    MAX_BACKOFF_S,
                    BASE_BACKOFF_S * (2 ** (attempt - 1)),
                )
                time.sleep(
                    min(
                        MAX_BACKOFF_S,
                        retry_after_seconds(exc, fallback),
                    )
                )

        raise last_exc or RuntimeError(
            f"Model call failed: {purpose}"
        )

    def _send_p3_pass(
        self,
        *,
        parts: list[PromptPart],
        purpose: str,
        sample_id: str,
        max_tokens: int,
        request_pass: str,
    ) -> dict:
        """
        Exact request-level scheduling used by the frozen Gemma P3 notebook.

        IMPORTANT:
        A main-pass failure is NOT immediately retried with the two-hour
        deferred timeout. It is returned to the cohort scheduler, which puts
        the sample into the deferred queue and revisits it only after the main
        pass reaches the end.
        """
        if request_pass == "main":
            active_client = self.client
            timeout_s = P3_REQUEST_TIMEOUT_S
            attempt_limit = P3_MAX_ATTEMPTS
        elif request_pass == "deferred":
            active_client = self.deferred_client
            timeout_s = P3_DEFERRED_REQUEST_TIMEOUT_S
            attempt_limit = P3_DEFERRED_MAX_ATTEMPTS
        else:
            raise ValueError(
                "P3 request_pass must be 'main' or 'deferred', "
                f"got {request_pass!r}."
            )

        if int(max_tokens) not in {
            P3_PLAN_MAX_OUTPUT_TOKENS,
            P3_ANSWER_MAX_OUTPUT_TOKENS,
        }:
            raise RuntimeError(
                f"Unapproved P3 max_tokens={max_tokens}; "
                "the frozen protocol uses 8192 for both calls."
            )

        inline_estimate = self._estimate_inline_bytes(parts)

        # The original P3 notebook used lossless inline PNG only and did not
        # switch to the Files API for large requests.
        if inline_estimate >= P3_INLINE_REQUEST_SAFETY_BYTES:
            raise RuntimeError(
                "P3_INLINE_SAFETY_GATE: estimated request size "
                f"{inline_estimate} bytes exceeds the frozen "
                f"{P3_INLINE_REQUEST_SAFETY_BYTES}-byte safety gate."
            )

        transport_mode = "inline_png"
        prompt_hash = parts_sha256(parts)
        last_exc = None

        for attempt in range(1, attempt_limit + 1):
            # Same pacing rule as the original notebook: at least five seconds
            # between the START times of provider requests.
            self._pace()
            self.last_request_started = time.monotonic()
            self.call_counter += 1

            started = time.perf_counter()
            event_base = {
                "sample_id": sample_id,
                "purpose": purpose,
                "pass": request_pass,
                "attempt": attempt,
                "model": MODEL,
                "temperature": TEMPERATURE,
                "seed": API_SEED,
                "max_output_tokens": int(max_tokens),
                "thinking_level": THINKING_LEVEL,
                "top_p_sent": False,
                "top_k_sent": False,
                "prompt_sha256": prompt_hash,
                "transport_mode": transport_mode,
                "inline_payload_estimate_bytes": inline_estimate,
                "p3_scheduling_version": P3_SCHEDULING_VERSION,
            }

            print(
                f"[P3 API START] sample={sample_id} | purpose={purpose} | "
                f"pass={request_pass} | attempt={attempt}/{attempt_limit} | "
                f"timeout_s={timeout_s} | "
                f"est_inline_mib={inline_estimate/(1024**2):.3f}",
                flush=True,
            )

            try:
                contents = self._to_google_contents(
                    parts,
                    active_client,
                    "inline_png",
                    event_base,
                )

                response = active_client.models.generate_content(
                    model=MODEL,
                    contents=contents,
                    config=self._generation_config(max_tokens),
                )

                elapsed = time.perf_counter() - started
                parsed = self._extract_response(response)

                reasoning_tokens = parsed["reasoning_tokens"]
                reasoning_violation = (
                    bool(parsed["provider_reasoning"].strip())
                    or (
                        reasoning_tokens is not None
                        and int(reasoning_tokens) > 0
                    )
                    or bool(
                        re.search(
                            r"</?think(?:\s[^>]*)?>",
                            parsed["text"],
                            flags=re.IGNORECASE,
                        )
                    )
                )

                if reasoning_violation:
                    raise RuntimeError(
                        "Provider-native reasoning evidence appeared "
                        "despite thinking_level='minimal'."
                    )

                result = {
                    "api_ok": True,
                    "text": parsed["text"],
                    "latency_s": elapsed,
                    "attempt": attempt,
                    "pass": request_pass,
                    "transport_mode": "inline_png",
                    "prompt_sha256": prompt_hash,
                    "prompt_tokens": parsed["prompt_tokens"],
                    "completion_tokens": parsed["completion_tokens"],
                    "reasoning_tokens": parsed["reasoning_tokens"],
                    "total_tokens": parsed["total_tokens"],
                    "finish_reason": parsed["finish_reason"],
                    "response_id": parsed["response_id"],
                    "model_version": parsed["model_version"],
                    "purpose": purpose,
                    "request_timeout_s": timeout_s,
                }

                append_jsonl(
                    API_EVENTS_PATH,
                    {
                        **event_base,
                        **result,
                        "event": "request_success",
                        "utc": datetime.now(timezone.utc).isoformat(),
                    },
                )

                print(
                    f"[P3 API OK] sample={sample_id} | purpose={purpose} | "
                    f"pass={request_pass} | attempt={attempt}/{attempt_limit} | "
                    f"elapsed_s={elapsed:.1f} | "
                    f"prompt_tokens={parsed['prompt_tokens']} | "
                    f"completion_tokens={parsed['completion_tokens']}",
                    flush=True,
                )
                return result

            except Exception as exc:
                last_exc = exc
                elapsed = time.perf_counter() - started
                status = status_from_exception(exc)
                retryable = (
                    status is None
                    or status in P3_RETRYABLE_STATUS
                    or isinstance(exc, TimeoutError)
                )
                will_retry = retryable and attempt < attempt_limit

                append_jsonl(
                    API_EVENTS_PATH,
                    {
                        **event_base,
                        "event": "request_error",
                        "latency_s": elapsed,
                        "http_status": status,
                        "will_retry": will_retry,
                        "error_type": type(exc).__name__,
                        "error_message": str(exc)[:8000],
                        "utc": datetime.now(timezone.utc).isoformat(),
                    },
                )

                print(
                    f"[P3 API ERROR] sample={sample_id} | purpose={purpose} | "
                    f"pass={request_pass} | attempt={attempt}/{attempt_limit} | "
                    f"elapsed_s={elapsed:.1f} | status={status} | "
                    f"retry={will_retry} | "
                    f"{type(exc).__name__}: {str(exc)[:300]}",
                    flush=True,
                )

                if not will_retry:
                    break

                fallback = min(
                    P3_MAX_BACKOFF_S,
                    P3_BASE_BACKOFF_S * (2 ** (attempt - 1)),
                )
                wait_s = min(
                    P3_MAX_BACKOFF_S,
                    retry_after_seconds(exc, fallback),
                )

                append_jsonl(
                    API_EVENTS_PATH,
                    {
                        **event_base,
                        "event": "retry_backoff",
                        "backoff_s": wait_s,
                        "utc": datetime.now(timezone.utc).isoformat(),
                    },
                )
                print(
                    f"[P3 BACKOFF] sample={sample_id} | "
                    f"purpose={purpose} | wait_s={wait_s:.1f}",
                    flush=True,
                )
                time.sleep(wait_s)

        raise last_exc or RuntimeError(
            f"P3 model call failed: {purpose}"
        )

    def send_p3(
        self,
        parts: list[PromptPart],
        purpose: str,
        sample_id: str,
        *,
        max_tokens: int,
        request_pass: str,
    ) -> dict:
        return self._send_p3_pass(
            parts=parts,
            purpose=purpose,
            sample_id=sample_id,
            max_tokens=max_tokens,
            request_pass=request_pass,
        )

    def send(
        self,
        parts: list[PromptPart],
        purpose: str,
        sample_id: str,
        max_tokens: int = MAX_OUTPUT_TOKENS,
    ) -> dict:
        try:
            return self._send_pass(
                parts=parts,
                purpose=purpose,
                sample_id=sample_id,
                max_tokens=max_tokens,
                active_client=self.client,
                timeout_s=REQUEST_TIMEOUT_S,
                attempt_limit=MAX_ATTEMPTS,
                pass_name="main",
            )
        except Exception as first_exc:
            print(
                f"[API DEFERRED] sample={sample_id} | purpose={purpose} | "
                f"main pass exhausted -> deferred timeout={DEFERRED_REQUEST_TIMEOUT_S}s "
                f"x {DEFERRED_MAX_ATTEMPTS} attempts",
                flush=True,
            )
            append_jsonl(
                API_EVENTS_PATH,
                {
                    "sample_id": sample_id,
                    "purpose": purpose,
                    "event": "enter_deferred_pass",
                    "error_type": type(first_exc).__name__,
                    "error_message": str(first_exc)[:4000],
                    "utc": datetime.now(timezone.utc).isoformat(),
                },
            )

            return self._send_pass(
                parts=parts,
                purpose=purpose,
                sample_id=sample_id,
                max_tokens=max_tokens,
                active_client=self.deferred_client,
                timeout_s=DEFERRED_REQUEST_TIMEOUT_S,
                attempt_limit=DEFERRED_MAX_ATTEMPTS,
                pass_name="deferred",
            )


print("Google API client class loaded.")

Google API client class loaded.


## 8. Domain classifier

The classifier uses the same multimodal model and a closed six-domain label set. Its output is parsed deterministically. A subject label is requested in the same call only to preserve the original same-subject demonstration rule when the Few-shot Complexity route is selected.

The subject field does **not** choose the prompting strategy; the strategy is determined exclusively by the resolved domain.

In [7]:
DOMAIN_CLASSIFIER_PROMPT = """Classify the academic domain of the multiple-choice problem above.

Choose exactly one domain from this closed set:
- Art and Design
- Business
- Health and Medicine
- Humanities and Social Science
- Science
- Tech and Engineering

Choose exactly one closest subject from this closed subject taxonomy:
- Accounting
- Agriculture
- Architecture_and_Engineering
- Art
- Art_Theory
- Basic_Medical_Science
- Biology
- Chemistry
- Clinical_Medicine
- Computer_Science
- Design
- Diagnostics_and_Laboratory_Medicine
- Economics
- Electronics
- Energy_and_Power
- Finance
- Geography
- History
- Literature
- Manage
- Marketing
- Materials
- Math
- Mechanical_Engineering
- Music
- Pharmacy
- Physics
- Psychology
- Public_Health
- Sociology

Return exactly one JSON object and no additional prose:
{"domain":"<DOMAIN>","subject":"<SUBJECT>"}"""


def build_classifier_parts(problem: Problem) -> list[PromptPart]:
    parts = []

    if RUNTIME_CONFIG["classifier_uses_images"]:
        parts.extend(
            image_prefix_parts(
                problem,
                include_labels=False,
            )
        )

    text = (
        f"{normalize_marker_text(problem.question)}\n\n"
        f"{normalize_marker_text(format_options(problem.options))}\n\n"
        f"{DOMAIN_CLASSIFIER_PROMPT}"
    )
    parts.append(PromptPart.text_part(text))
    return parts


def normalize_label(value: Any) -> Optional[str]:
    if value is None:
        return None
    return unicodedata.normalize(
        "NFKC",
        str(value),
    ).strip()


def parse_domain_classifier(text: str) -> dict:
    raw = str(text or "").strip()

    # Prefer a JSON object.
    candidates = re.findall(r"\{.*?\}", raw, flags=re.DOTALL)

    for candidate in candidates:
        try:
            payload = json.loads(candidate)
        except Exception:
            continue

        domain = normalize_label(payload.get("domain"))
        subject = normalize_label(payload.get("subject"))

        if (
            domain in VALID_DOMAINS
            and subject in VALID_SUBJECTS
            and SUBJECT_TO_DOMAIN[subject] == domain
        ):
            return {
                "domain": domain,
                "subject": subject,
                "parse_method": "json",
            }

    # Strict deterministic fallback over exact known labels.
    domain_hits = [
        domain
        for domain in VALID_DOMAINS
        if re.search(
            rf"(?<!\w){re.escape(domain)}(?!\w)",
            raw,
            flags=re.IGNORECASE,
        )
    ]

    if len(domain_hits) != 1:
        raise ValueError(
            "Could not recover exactly one valid domain from classifier output."
        )

    domain = domain_hits[0]

    subject_hits = [
        subject
        for subject in VALID_SUBJECTS
        if re.search(
            rf"(?<!\w){re.escape(subject)}(?!\w)",
            raw,
            flags=re.IGNORECASE,
        )
    ]

    valid_subject_hits = [
        subject
        for subject in subject_hits
        if SUBJECT_TO_DOMAIN[subject] == domain
    ]

    if len(valid_subject_hits) != 1:
        raise ValueError(
            "Could not recover exactly one compatible subject from classifier output."
        )

    return {
        "domain": domain,
        "subject": valid_subject_hits[0],
        "parse_method": "closed_label_fallback",
    }


def valid_context_route(
    context: Optional[dict],
) -> Optional[dict]:
    if not context:
        return None

    domain = normalize_label(context.get("domain"))
    subject = normalize_label(context.get("subject"))

    if domain not in VALID_DOMAINS and subject in VALID_SUBJECTS:
        domain = SUBJECT_TO_DOMAIN[subject]

    if domain not in VALID_DOMAINS:
        return None

    if subject not in VALID_SUBJECTS:
        subject = None

    if (
        subject is not None
        and SUBJECT_TO_DOMAIN[subject] != domain
    ):
        subject = None

    return {
        "domain": domain,
        "subject": subject,
        "route_source": "record_context",
        "classifier_raw": None,
        "classifier_parse_method": None,
    }


print("Domain classifier loaded.")

Domain classifier loaded.


## 9. Trained-classifier contract

A future trained classifier is intentionally decoupled from the router architecture. It must implement:

```python
predict(question=..., images=..., options=...) -> {
    "domain": "...",
    "subject": "..."
}
```

The subject output is retained because the frozen Few-shot Complexity strategy uses a same-subject demonstration pool. A reference Hugging Face adapter is included; a multimodal trained classifier can implement the same protocol directly without changing the router.


In [8]:
class TrainedClassifierProtocol(Protocol):
    def predict(
        self,
        *,
        question: str,
        images: list[Image.Image],
        options: list[str],
    ) -> dict:
        ...


class HuggingFaceTextClassifier:
    """Reference adapter for two locally trained sequence classifiers."""

    def __init__(
        self,
        *,
        domain_model_path: str,
        subject_model_path: str,
        device: Optional[str] = None,
        max_length: int = 512,
    ):
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.max_length = int(max_length)

        self.domain_tokenizer = AutoTokenizer.from_pretrained(domain_model_path)
        self.domain_model = AutoModelForSequenceClassification.from_pretrained(
            domain_model_path
        ).to(self.device).eval()

        self.subject_tokenizer = AutoTokenizer.from_pretrained(subject_model_path)
        self.subject_model = AutoModelForSequenceClassification.from_pretrained(
            subject_model_path
        ).to(self.device).eval()

        self._validate_labels(self.domain_model.config.id2label, set(VALID_DOMAINS), "domain")
        self._validate_labels(self.subject_model.config.id2label, set(VALID_SUBJECTS), "subject")

    @staticmethod
    def _validate_labels(id2label, expected, name):
        observed = {str(v) for v in id2label.values()}
        if observed != expected:
            raise RuntimeError(
                f"{name} classifier label-space mismatch. "
                f"Expected={sorted(expected)}, observed={sorted(observed)}"
            )

    def _predict_one(self, text, tokenizer, model):
        encoded = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length,
        )
        encoded = {k: v.to(self.device) for k, v in encoded.items()}
        with self.torch.no_grad():
            logits = model(**encoded).logits[0]
        probs = self.torch.softmax(logits, dim=-1)
        idx = int(probs.argmax().item())
        return str(model.config.id2label[idx]), float(probs[idx].item())

    def predict(self, *, question, images, options):
        text = f"{question}\n\n{format_options(options)}"
        domain, domain_conf = self._predict_one(
            text, self.domain_tokenizer, self.domain_model
        )
        subject, subject_conf = self._predict_one(
            text, self.subject_tokenizer, self.subject_model
        )
        if domain not in VALID_DOMAINS or subject not in VALID_SUBJECTS:
            raise RuntimeError("Trained classifier returned an unknown label.")
        if SUBJECT_TO_DOMAIN[subject] != domain:
            raise RuntimeError(
                f"Hierarchically inconsistent prediction: domain={domain}, subject={subject}"
            )
        return {
            "domain": domain,
            "subject": subject,
            "domain_confidence": domain_conf,
            "subject_confidence": subject_conf,
            "classifier_type": "hf_text_dual_classifier",
        }


print("Trained-classifier interface ready.")

Trained-classifier interface ready.


## 10. Pinned MMMU-Pro resolver

The benchmark resolver uses the same pinned `MMMU/MMMU_Pro` Standard (10 options) test revision as the controlled experiments. A sample is addressed by exact ID; the row's published subject is mapped to the six project domains through the frozen taxonomy.


In [9]:
MMMU_PRO_REPO = "MMMU/MMMU_Pro"
MMMU_PRO_CONFIG = "standard (10 options)"
MMMU_PRO_SPLIT = "test"
MMMU_PRO_REVISION = "563f3e84bb3b90893083a1f039cfa13077f2302b"
MMMU_PRO_EXPECTED_N = 1730


class MMMUProResolver:
    def __init__(self):
        self.dataset = None
        self.index_by_id = None

    def _ensure_loaded(self):
        if self.dataset is not None:
            return
        from datasets import load_dataset
        ds = load_dataset(
            MMMU_PRO_REPO,
            MMMU_PRO_CONFIG,
            split=MMMU_PRO_SPLIT,
            revision=MMMU_PRO_REVISION,
        )
        if len(ds) != MMMU_PRO_EXPECTED_N:
            raise RuntimeError(f"Unexpected MMMU-Pro size: {len(ds)}")
        ids = [str(x) for x in ds["id"]]
        if len(set(ids)) != len(ids):
            raise RuntimeError("MMMU-Pro IDs are not unique.")
        self.dataset = ds
        self.index_by_id = {sid: i for i, sid in enumerate(ids)}

    def row(self, sample_id: str) -> dict:
        self._ensure_loaded()
        sample_id = str(sample_id)
        if sample_id not in self.index_by_id:
            raise KeyError(f"Unknown MMMU-Pro ID: {sample_id}")
        return self.dataset[self.index_by_id[sample_id]]

    @staticmethod
    def row_images(row: dict) -> list[Image.Image]:
        indexed = []
        for i in range(1, 64):
            key = f"image_{i}"
            if key not in row:
                break
            if row.get(key) is not None:
                indexed.append((i, image_to_pil(row[key])))
        if not indexed:
            raise RuntimeError(f"No image found for MMMU-Pro row {row.get('id')}")
        observed = [i for i, _ in indexed]
        expected = list(range(1, max(observed) + 1))
        if observed != expected:
            raise RuntimeError(f"Non-contiguous image columns: {observed}")
        return [image for _, image in indexed]

    def problem(self, sample_id: str) -> Problem:
        row = self.row(sample_id)
        subject = str(row["subject"])
        if subject not in SUBJECT_TO_DOMAIN:
            raise RuntimeError(f"Unmapped MMMU-Pro subject: {subject}")
        return build_problem(
            question=str(row["question"]),
            options=parse_options_value(row["options"]),
            images=self.row_images(row),
            sample_id=str(sample_id),
            subject=subject,
            domain=SUBJECT_TO_DOMAIN[subject],
            difficulty=str(row["topic_difficulty"]).strip().title(),
        )


print("MMMU-Pro resolver ready.")

MMMU-Pro resolver ready.


## 11. Few-shot Complexity retrieval: frozen demonstrations + lazy runtime query embeddings

The final router uses the fixed 405-item demonstration bank and its historical
precomputed RICES embeddings. The current query is **not embedded immediately**.

### Lazy retrieval policy

For a query routed to Few-shot Complexity, candidates are first restricted to
the same subject and ranked using metadata only:

1. `complexity_n_steps`, descending;
2. difficulty priority `Hard > Medium > Easy`.

The router then walks these priority groups until the Top-3 membership is
determined.

- If a complete `(STEP count, difficulty)` group fits inside the remaining
  Top-3 slots, that group is accepted **without computing query embeddings**.
- If a group is larger than the number of remaining Top-3 slots, membership is
  ambiguous. Only then is the current query embedded with the pinned CLIP
  encoder, and cosine similarity is computed **only for that boundary tie
  group**.
- Historical query embeddings are never used.
- Only the three finally selected multimodal demonstrations are loaded for
  prompt construction.

Thus CLIP is not invoked for a question when STEP count and difficulty already
determine the three demonstration IDs.

### Conditional cosine protocol

When required, the current query is embedded using the same RICES protocol as
the frozen demonstration corpus:

- `openai/clip-vit-large-patch14`;
- revision `32bd64288804d66eefd0ccbe215aa642df71cc41`;
- model-weight SHA-256
  `a2bf730a0c7debf160f7a6b50b3aaf3703e7e88ac73de7a314903141db026dcb`;
- question stem only, with literal `<image n>` markers removed;
- options, answer labels, explanations, and CoT excluded;
- CLIP text context length 77;
- float32 inference;
- EXIF orientation correction and RGB conversion for images;
- unit-L2 normalization for present modalities.

For the boundary tie group:

\[
s(d,q)=\cos(T_d,T_q)+
\operatorname{mean}_{i,j}\cos(I_{q,i}, I_{d,j}).
\]

Ties remaining after cosine are broken by frozen demonstration order and then
demonstration ID.

The final three selected demonstrations are inserted in reverse selected order
so the highest-priority example is closest to the final query.

### Transformers return-type compatibility

The runtime encoder explicitly supports both CLIP API behaviors encountered
across Transformers releases: a direct projected tensor and a
`BaseModelOutputWithPooling` object. In the latter case, the projected
768-dimensional feature is read from `pooler_output`. Both text and image
features are shape- and finiteness-checked before L2 normalization.

In [10]:
DEMO_REPO = "MMMU/MMMU"
DEMO_REVISION = "98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68"
DEMO_EXPECTED_N = 405
DEMO_CSV_NAME = (
    "cot_pipeline_clean_merged_KEEP_only_grouped_by_subject.csv"
)
EXPECTED_DEMO_CSV_SHA256 = (
    "2c55994efcbaf2d1e21e43f256a70e3e10859f6f20eafb68c41a48d9ba2666ca"
)
EXPECTED_DEMO_IDS_CANONICAL_SHA256 = (
    "92bcccab502feaf63e61cd3c49cae591cd41b3c310a8eae0cccef8a814c2a635"
)

CLIP_REPO = "openai/clip-vit-large-patch14"
CLIP_REVISION = "32bd64288804d66eefd0ccbe215aa642df71cc41"
CLIP_EXPECTED_MODEL_SHA256 = (
    "a2bf730a0c7debf160f7a6b50b3aaf3703e7e88ac73de7a314903141db026dcb"
)
CLIP_DIM = 768
CLIP_EXPECTED_CONTEXT = 77

EXPECTED_EMBEDDING_PROTOCOL = (
    "RICES_MMMU_MMMUPRO_CLIP_VITL14_EMBEDDINGS_V1_1_20260905"
)
DEMO_EMBEDDING_FILES = (
    "demonstration_question_embeddings.parquet",
    "demonstration_image_embeddings.parquet",
)
FORBIDDEN_EMBEDDING_COLUMNS = {
    "options", "options_json", "answer", "gold_answer", "explanation",
    "original_explanation", "generated_cot", "cot", "call1_raw_response",
}

RICES_IMAGE_REF_RE = re.compile(
    r"<image\s*([1-7])\s*>",
    flags=re.IGNORECASE,
)
RICES_WHITESPACE_RE = re.compile(r"\s+")

SHOTS = 3
DIFFICULTY_PRIORITY = {
    "Easy": 1,
    "Medium": 2,
    "Hard": 3,
}

CLIP_MODEL_CACHE_DIR = (
    CACHE_DIR / "clip_vitl14_32bd642"
)
CLIP_MODEL_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def sha_file(
    path: Path,
    chunk_size: int = 16 * 1024 * 1024,
) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(chunk_size),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_id_sha256(ids) -> str:
    payload = "\n".join(
        str(value)
        for value in ids
    ) + "\n"
    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


def normalize_question_for_rices(
    question: str,
) -> str:
    without_markers = RICES_IMAGE_REF_RE.sub(
        " ",
        str(question),
    )
    return RICES_WHITESPACE_RE.sub(
        " ",
        without_markers,
    ).strip()


def discover_demo_csv() -> Path:
    roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
        Path("."),
    ]

    matches = set()

    for root in roots:
        if not root.exists():
            continue

        exact = root / DEMO_CSV_NAME
        if exact.is_file():
            matches.add(exact.resolve())

        for candidate in root.rglob(
            "cot_pipeline_clean_merged_KEEP_only_grouped_by_subject*.csv"
        ):
            if candidate.is_file():
                matches.add(candidate.resolve())

    valid = [
        path
        for path in sorted(matches)
        if sha_file(path) == EXPECTED_DEMO_CSV_SHA256
    ]

    if not valid:
        raise FileNotFoundError(
            "The validated 405-row demonstration CSV was not found. "
            f"Attach '{DEMO_CSV_NAME}' to the runtime. "
            f"Expected SHA-256: {EXPECTED_DEMO_CSV_SHA256}"
        )

    return valid[0]


def extract_clip_projected_features(
    output,
    *,
    modality: str,
):
    """
    Normalize Hugging Face CLIP get_*_features return types.

    Compatible with:
      - historical Transformers behavior: torch.Tensor
      - newer behavior: BaseModelOutputWithPooling whose pooler_output
        contains the projected CLIP embedding.

    Returns a 2D torch.Tensor of shape [batch, CLIP_DIM].
    """
    import torch

    tensor = None

    # Historical Transformers: direct projected tensor.
    if torch.is_tensor(output):
        tensor = output

    # Newer Transformers: projected feature stored in pooler_output.
    elif (
        hasattr(output, "pooler_output")
        and torch.is_tensor(
            getattr(output, "pooler_output")
        )
    ):
        tensor = output.pooler_output

    # Defensive support for dict-like outputs.
    elif isinstance(output, dict):
        pooled = output.get("pooler_output")
        if torch.is_tensor(pooled):
            tensor = pooled

    # Defensive support for tuple/list return mode.
    elif isinstance(output, (tuple, list)):
        candidates = [
            value
            for value in output
            if (
                torch.is_tensor(value)
                and value.ndim == 2
                and value.shape[-1] == CLIP_DIM
            )
        ]
        if len(candidates) == 1:
            tensor = candidates[0]

    if tensor is None:
        raise TypeError(
            "Unsupported CLIP "
            f"{modality} feature output type: "
            f"{type(output).__name__}. "
            "Expected a Tensor or an object with a Tensor "
            "pooler_output."
        )

    if tensor.ndim != 2:
        raise RuntimeError(
            f"CLIP {modality} projected feature must be 2D; "
            f"observed shape={tuple(tensor.shape)}."
        )

    if int(tensor.shape[-1]) != CLIP_DIM:
        raise RuntimeError(
            f"CLIP {modality} projected feature dimension mismatch: "
            f"{int(tensor.shape[-1])} != {CLIP_DIM}."
        )

    if not torch.isfinite(tensor).all().item():
        raise RuntimeError(
            f"CLIP {modality} projected feature contains non-finite values."
        )

    return tensor


class HybridRICESRetriever:
    """
    Runtime query embeddings + frozen historical demonstration embeddings.

    The historical development-query embeddings are never used.
    """

    def __init__(self):
        self.model = None
        self.processor = None
        self.device = None
        self.clip_context = None

        self.query_embedding_cache = {}

        self.demo_csv_path = discover_demo_csv()
        self.demo_table = pd.read_csv(
            self.demo_csv_path,
            low_memory=False,
        )
        self.demo_ids = (
            self.demo_table["question_id"]
            .astype(str)
            .tolist()
        )

        if (
            len(self.demo_ids) != DEMO_EXPECTED_N
            or len(set(self.demo_ids)) != DEMO_EXPECTED_N
        ):
            raise RuntimeError(
                "RICES demonstration-ID cardinality mismatch."
            )

        if (
            canonical_id_sha256(self.demo_ids)
            != EXPECTED_DEMO_IDS_CANONICAL_SHA256
        ):
            raise RuntimeError(
                "RICES demonstration-ID order/hash mismatch."
            )

        (
            self.demo_bundle_signature,
            self.demo_embedding_root,
            self.demo_embedding_manifest,
            self.demo_embedding_validation,
            self.demo_embedding_hashes,
        ) = self._discover_frozen_demo_embeddings()

        self._load_frozen_demo_embeddings()

        print(
            "Frozen demonstration embeddings validated:",
            self.demo_embedding_root,
        )
        print(
            "Frozen demo embedding signature:",
            self.demo_bundle_signature,
        )

    def _discover_frozen_demo_embeddings(self):
        candidates = []

        for root in (
            Path("/kaggle/input"),
            Path("/kaggle/working"),
        ):
            if not root.exists():
                continue

            for manifest_path in root.rglob(
                "embedding_manifest.json"
            ):
                base = manifest_path.parent
                final_path = (
                    base / "FINAL_VALIDATION.json"
                )

                if (
                    not final_path.is_file()
                    or not all(
                        (base / name).is_file()
                        for name in DEMO_EMBEDDING_FILES
                    )
                ):
                    continue

                try:
                    manifest = json.loads(
                        manifest_path.read_text(
                            encoding="utf-8"
                        )
                    )
                    final = json.loads(
                        final_path.read_text(
                            encoding="utf-8"
                        )
                    )
                except Exception:
                    continue

                if (
                    manifest.get("status") != "PASS"
                    or final.get("status") != "PASS"
                ):
                    continue

                if (
                    manifest.get("protocol_version")
                    != EXPECTED_EMBEDDING_PROTOCOL
                    or final.get("protocol_version")
                    != EXPECTED_EMBEDDING_PROTOCOL
                ):
                    continue

                demo_source = (
                    manifest.get(
                        "demonstration_source"
                    )
                    or {}
                )
                if (
                    demo_source.get("repo")
                    != DEMO_REPO
                    or demo_source.get("revision")
                    != DEMO_REVISION
                    or int(
                        demo_source.get(
                            "selected_rows",
                            -1,
                        )
                    ) != DEMO_EXPECTED_N
                    or demo_source.get(
                        "selection_csv_sha256"
                    ) != EXPECTED_DEMO_CSV_SHA256
                    or demo_source.get(
                        "selected_ids_canonical_sha256"
                    ) != EXPECTED_DEMO_IDS_CANONICAL_SHA256
                ):
                    continue

                model_info = (
                    manifest.get(
                        "embedding_model"
                    )
                    or {}
                )
                if (
                    model_info.get("repo")
                    != CLIP_REPO
                    or model_info.get("revision")
                    != CLIP_REVISION
                    or model_info.get(
                        "model_safetensors_sha256"
                    ) != CLIP_EXPECTED_MODEL_SHA256
                    or int(
                        model_info.get(
                            "dimension",
                            -1,
                        )
                    ) != CLIP_DIM
                    or int(
                        model_info.get(
                            "text_context_length",
                            -1,
                        )
                    ) != CLIP_EXPECTED_CONTEXT
                    or model_info.get(
                        "output_dtype"
                    ) != "float32"
                ):
                    continue

                semantic = (
                    manifest.get(
                        "semantic_input"
                    )
                    or {}
                )
                if not (
                    semantic.get("text")
                    == "question stem only"
                    and semantic.get(
                        "options_embedded"
                    ) is False
                    and semantic.get(
                        "answers_embedded"
                    ) is False
                    and semantic.get(
                        "explanations_embedded"
                    ) is False
                    and semantic.get(
                        "cot_embedded"
                    ) is False
                    and semantic.get(
                        "multi_image_aggregation"
                    ) is None
                ):
                    continue

                hashes = {
                    name: sha_file(
                        base / name
                    )
                    for name in DEMO_EMBEDDING_FILES
                }

                artifacts = (
                    manifest.get(
                        "artifacts"
                    )
                    or {}
                )
                if any(
                    (
                        artifacts.get(name)
                        or {}
                    ).get("sha256")
                    != hashes[name]
                    for name
                    in DEMO_EMBEDDING_FILES
                ):
                    continue

                signature = hashlib.sha256(
                    json.dumps(
                        hashes,
                        sort_keys=True,
                        separators=(",", ":"),
                    ).encode("utf-8")
                ).hexdigest()

                candidates.append(
                    (
                        signature,
                        base,
                        manifest,
                        final,
                        hashes,
                    )
                )

        if not candidates:
            raise RuntimeError(
                "No valid historical demonstration-embedding bundle "
                "was found. Attach the complete PASS output of the "
                "original RICES embedding notebook containing "
                "demonstration_question_embeddings.parquet, "
                "demonstration_image_embeddings.parquet, "
                "embedding_manifest.json, and FINAL_VALIDATION.json."
            )

        signatures = sorted({
            item[0]
            for item in candidates
        })

        if len(signatures) != 1:
            raise RuntimeError(
                "Conflicting historical demonstration embeddings "
                f"were found: {signatures}"
            )

        candidates.sort(
            key=lambda item: str(item[1])
        )
        return candidates[0]

    def _load_demo_table(
        self,
        name: str,
        expected_rows: int,
        modality: str,
    ):
        path = (
            self.demo_embedding_root
            / name
        )
        table = pq.read_table(path)

        if table.num_rows != expected_rows:
            raise RuntimeError(
                f"{name}: row count "
                f"{table.num_rows} != {expected_rows}"
            )

        if "embedding" not in table.column_names:
            raise RuntimeError(
                f"{name}: embedding column missing"
            )

        leaked = (
            FORBIDDEN_EMBEDDING_COLUMNS
            & set(table.column_names)
        )
        if leaked:
            raise RuntimeError(
                f"{name}: forbidden semantic fields leaked: "
                f"{sorted(leaked)}"
            )

        vectors = np.asarray(
            table.column(
                "embedding"
            ).to_pylist(),
            dtype=np.float32,
        )

        if vectors.shape != (
            expected_rows,
            CLIP_DIM,
        ):
            raise RuntimeError(
                f"{name}: vector shape mismatch "
                f"{vectors.shape}"
            )

        if not np.isfinite(vectors).all():
            raise RuntimeError(
                f"{name}: non-finite vector"
            )

        metadata = (
            table.drop(
                ["embedding"]
            )
            .to_pandas()
        )

        schema_metadata = (
            table.schema.metadata
            or {}
        )
        expected_schema = {
            b"protocol_version": (
                EXPECTED_EMBEDDING_PROTOCOL.encode()
            ),
            b"embedding_model": (
                CLIP_REPO.encode()
            ),
            b"embedding_revision": (
                CLIP_REVISION.encode()
            ),
            b"multi_image_aggregation": b"none",
        }

        for key, expected in (
            expected_schema.items()
        ):
            if (
                schema_metadata.get(key)
                != expected
            ):
                raise RuntimeError(
                    f"{name}: schema metadata mismatch "
                    f"for {key!r}"
                )

        norms = np.linalg.norm(
            vectors,
            axis=1,
        )

        if modality == "question":
            if (
                "text_similarity_eligible"
                not in metadata.columns
            ):
                raise RuntimeError(
                    f"{name}: missing text-similarity flag"
                )
            eligible = (
                metadata[
                    "text_similarity_eligible"
                ]
                .astype(bool)
                .to_numpy()
            )
            expected_norms = (
                eligible.astype(np.float32)
            )
        else:
            expected_norms = np.ones(
                expected_rows,
                dtype=np.float32,
            )

        max_norm_error = float(
            np.max(
                np.abs(
                    norms
                    - expected_norms
                )
            )
        )
        if max_norm_error >= 2e-5:
            raise RuntimeError(
                f"{name}: vector norm policy mismatch "
                f"({max_norm_error})"
            )

        manifest_entry = (
            self.demo_embedding_manifest
            .get("artifacts", {})
            .get(name, {})
        )

        if (
            manifest_entry.get("sha256")
            != self.demo_embedding_hashes[
                name
            ]
        ):
            raise RuntimeError(
                f"{name}: manifest/file SHA mismatch"
            )

        return metadata, vectors

    def _load_frozen_demo_embeddings(self):
        manifest_artifacts = (
            self.demo_embedding_manifest[
                "artifacts"
            ]
        )

        demo_image_rows = int(
            manifest_artifacts[
                "demonstration_image_embeddings.parquet"
            ]["rows"]
        )

        (
            self.demo_text_meta,
            self.demo_text_vectors,
        ) = self._load_demo_table(
            "demonstration_question_embeddings.parquet",
            DEMO_EXPECTED_N,
            "question",
        )

        (
            self.demo_image_meta,
            self.demo_image_vectors,
        ) = self._load_demo_table(
            "demonstration_image_embeddings.parquet",
            demo_image_rows,
            "image",
        )

        embedded_demo_ids = (
            self.demo_text_meta[
                "sample_id"
            ]
            .astype(str)
            .tolist()
        )

        if (
            embedded_demo_ids
            != self.demo_ids
        ):
            raise RuntimeError(
                "Frozen demo text embedding ID order mismatch."
            )

        if (
            self.demo_text_meta[
                "subject"
            ]
            .astype(str)
            .tolist()
            != self.demo_table[
                "subject"
            ]
            .astype(str)
            .tolist()
        ):
            raise RuntimeError(
                "Frozen demo embedding subject mismatch."
            )

        if (
            self.demo_text_meta[
                "question_raw"
            ]
            .astype(str)
            .tolist()
            != self.demo_table[
                "question"
            ]
            .astype(str)
            .tolist()
        ):
            raise RuntimeError(
                "Frozen demo embedding question mismatch."
            )

        if set(
            self.demo_image_meta[
                "sample_id"
            ].astype(str)
        ) != set(
            self.demo_ids
        ):
            raise RuntimeError(
                "Frozen demo image parent coverage mismatch."
            )

        self.demo_text_index = {
            sample_id: i
            for i, sample_id
            in enumerate(
                embedded_demo_ids
            )
        }

        self.demo_image_indices = {}
        for i, sample_id in enumerate(
            self.demo_image_meta[
                "sample_id"
            ].astype(str)
        ):
            self.demo_image_indices.setdefault(
                sample_id,
                [],
            ).append(i)

    def _ensure_query_encoder(self):
        if self.model is not None:
            return

        import torch
        from transformers import (
            CLIPModel,
            CLIPProcessor,
        )

        random.seed(API_SEED)
        np.random.seed(API_SEED)
        torch.manual_seed(API_SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(
                API_SEED
            )

        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.use_deterministic_algorithms(
            True,
            warn_only=False,
        )
        torch.set_float32_matmul_precision(
            "highest"
        )

        self.device = torch.device(
            "cuda:0"
            if torch.cuda.is_available()
            else "cpu"
        )

        model_snapshot = Path(
            snapshot_download(
                repo_id=CLIP_REPO,
                repo_type="model",
                revision=CLIP_REVISION,
                local_dir=str(
                    CLIP_MODEL_CACHE_DIR
                ),
            )
        )

        model_weights = (
            model_snapshot
            / "model.safetensors"
        )
        if not model_weights.is_file():
            raise RuntimeError(
                "Pinned CLIP model.safetensors was not downloaded."
            )

        observed_model_sha = sha_file(
            model_weights
        )
        if (
            observed_model_sha
            != CLIP_EXPECTED_MODEL_SHA256
        ):
            raise RuntimeError(
                "Pinned CLIP model weight SHA mismatch: "
                f"{observed_model_sha}"
            )

        self.processor = (
            CLIPProcessor.from_pretrained(
                str(model_snapshot),
                local_files_only=True,
            )
        )

        self.model = (
            CLIPModel.from_pretrained(
                str(model_snapshot),
                local_files_only=True,
                use_safetensors=True,
                torch_dtype=torch.float32,
            )
            .to(self.device)
        )
        self.model.eval()

        self.clip_context = int(
            self.processor.tokenizer.model_max_length
        )
        clip_dim = int(
            self.model.config.projection_dim
        )

        if (
            self.clip_context
            != CLIP_EXPECTED_CONTEXT
        ):
            raise RuntimeError(
                "Unexpected CLIP text context: "
                f"{self.clip_context}"
            )

        if clip_dim != CLIP_DIM:
            raise RuntimeError(
                "Unexpected CLIP projection dimension: "
                f"{clip_dim}"
            )

        print(
            "Runtime query encoder loaded:",
            f"{CLIP_REPO}@{CLIP_REVISION}",
            "| device:",
            self.device,
        )

    def _encode_query_text(
        self,
        raw_question: str,
    ) -> np.ndarray:
        self._ensure_query_encoder()

        import torch

        text = normalize_question_for_rices(
            raw_question
        )

        if not text:
            return np.zeros(
                CLIP_DIM,
                dtype=np.float32,
            )

        encoded = self.processor(
            text=[text],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.clip_context,
        )
        encoded = {
            key: value.to(
                self.device
            )
            for key, value
            in encoded.items()
        }

        with torch.inference_mode():
            raw_output = (
                self.model
                .get_text_features(
                    **encoded
                )
            )
            features = (
                extract_clip_projected_features(
                    raw_output,
                    modality="text",
                )
                .float()
            )
            raw_norm = (
                torch.linalg.vector_norm(
                    features,
                    dim=1,
                )
            )

            if (
                not torch.isfinite(
                    features
                ).all().item()
                or not (
                    raw_norm > 0
                ).all().item()
            ):
                raise RuntimeError(
                    "Invalid runtime CLIP text feature."
                )

            normalized = (
                features
                / raw_norm.unsqueeze(1)
            )

        vector = (
            normalized[0]
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=False,
            )
        )

        if (
            abs(
                float(
                    np.linalg.norm(
                        vector
                    )
                )
                - 1.0
            )
            >= 2e-5
        ):
            raise RuntimeError(
                "Runtime query text vector failed L2 audit."
            )

        return vector

    def _encode_query_images(
        self,
        images: list[ImageAsset],
    ) -> list[np.ndarray]:
        self._ensure_query_encoder()

        import torch

        vectors = []

        for image_asset in images:
            with Image.open(
                image_asset.rices_path
            ) as opened:
                opened.load()
                image = opened.convert(
                    "RGB"
                )

            encoded = self.processor(
                images=[image],
                return_tensors="pt",
            )

            pixel_values = (
                encoded["pixel_values"]
                .to(
                    self.device,
                    dtype=torch.float32,
                )
            )

            with torch.inference_mode():
                raw_output = (
                    self.model
                    .get_image_features(
                        pixel_values=pixel_values
                    )
                )
                features = (
                    extract_clip_projected_features(
                        raw_output,
                        modality="image",
                    )
                    .float()
                )
                raw_norm = (
                    torch.linalg.vector_norm(
                        features,
                        dim=1,
                    )
                )

                if (
                    not torch.isfinite(
                        features
                    ).all().item()
                    or not (
                        raw_norm > 0
                    ).all().item()
                ):
                    raise RuntimeError(
                        "Invalid runtime CLIP image feature."
                    )

                normalized = (
                    features
                    / raw_norm.unsqueeze(1)
                )

            vector = (
                normalized[0]
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            if (
                abs(
                    float(
                        np.linalg.norm(
                            vector
                        )
                    )
                    - 1.0
                )
                >= 2e-5
            ):
                raise RuntimeError(
                    "Runtime query image vector failed L2 audit."
                )

            vectors.append(vector)

        if not vectors:
            raise RuntimeError(
                "RICES retrieval requires at least one query image."
            )

        return vectors

    @staticmethod
    def _dot(
        a: np.ndarray,
        b: np.ndarray,
    ) -> float:
        a64 = np.asarray(
            a,
            dtype=np.float64,
        )
        b64 = np.asarray(
            b,
            dtype=np.float64,
        )

        if (
            a64.shape != (CLIP_DIM,)
            or b64.shape != (CLIP_DIM,)
        ):
            raise RuntimeError(
                "Cosine vector dimension mismatch."
            )

        return float(
            math.fsum(
                float(x) * float(y)
                for x, y in zip(
                    a64,
                    b64,
                )
            )
        )

    def query_embedding(
        self,
        problem: Problem,
    ):
        cache_key = (
            str(problem.sample_id),
            hashlib.sha256(
                str(problem.question)
                .encode("utf-8")
            ).hexdigest(),
            tuple(
                image.sha256
                for image in problem.images
            ),
        )

        if (
            cache_key
            not in self.query_embedding_cache
        ):
            _qe_t0 = time.perf_counter()
            print(
                f"[RICES QUERY START] sample={problem.sample_id} | "
                f"images={len(problem.images)}",
                flush=True,
            )

            _text_t0 = time.perf_counter()
            _q_text = self._encode_query_text(
                problem.question
            )
            print(
                f"[RICES TEXT OK] sample={problem.sample_id} | "
                f"elapsed_s={time.perf_counter() - _text_t0:.2f}",
                flush=True,
            )

            _img_t0 = time.perf_counter()
            _q_images = self._encode_query_images(
                problem.images
            )
            print(
                f"[RICES IMAGES OK] sample={problem.sample_id} | "
                f"count={len(_q_images)} | "
                f"elapsed_s={time.perf_counter() - _img_t0:.2f}",
                flush=True,
            )

            self.query_embedding_cache[
                cache_key
            ] = (
                _q_text,
                _q_images,
            )

            print(
                f"[RICES QUERY DONE] sample={problem.sample_id} | "
                f"elapsed_s={time.perf_counter() - _qe_t0:.2f}",
                flush=True,
            )

        return self.query_embedding_cache[
            cache_key
        ]

    def demo_embedding(
        self,
        demo: dict,
    ) -> str:
        demo_id = str(
            demo["sample_id"]
        )

        if (
            demo_id
            not in self.demo_text_index
        ):
            raise RuntimeError(
                "Frozen demonstration embedding missing: "
                f"{demo_id}"
            )

        metadata_row = (
            self.demo_text_meta.iloc[
                self.demo_text_index[
                    demo_id
                ]
            ]
        )

        if (
            str(
                metadata_row[
                    "question_raw"
                ]
            )
            != str(
                demo["problem"].question
            )
        ):
            raise RuntimeError(
                "Frozen demo/query source question mismatch: "
                f"{demo_id}"
            )

        if (
            str(
                metadata_row["subject"]
            )
            != str(
                demo["subject"]
            )
        ):
            raise RuntimeError(
                "Frozen demo subject mismatch: "
                f"{demo_id}"
            )

        return demo_id

    def similarity(
        self,
        query_embedding,
        demo_id: str,
    ) -> dict:
        q_text, q_images = (
            query_embedding
        )

        d_text = (
            self.demo_text_vectors[
                self.demo_text_index[
                    demo_id
                ]
            ]
        )

        text_score = self._dot(
            q_text,
            d_text,
        )

        pair_scores = []

        for q_image in q_images:
            for demo_index in (
                self.demo_image_indices[
                    demo_id
                ]
            ):
                pair_scores.append(
                    self._dot(
                        q_image,
                        self.demo_image_vectors[
                            demo_index
                        ],
                    )
                )

        if not pair_scores:
            raise RuntimeError(
                "Missing query/demo image pair for "
                f"demo={demo_id}"
            )

        image_score = float(
            math.fsum(
                pair_scores
            )
            / len(pair_scores)
        )

        return {
            "text_score": (
                text_score
            ),
            "image_score": (
                image_score
            ),
            "total_score": float(
                text_score
                + image_score
            ),
            "image_pair_count": (
                len(pair_scores)
            ),
        }


class DemoBank:
    def __init__(self):
        self.csv_path = None
        self.table = None
        self.source_snapshot = None
        self.parquet_cache = {}
        self.demo_cache = {}

    def _ensure_table(self):
        if self.table is not None:
            return

        self.csv_path = discover_demo_csv()
        table = pd.read_csv(self.csv_path)

        required = {
            "subject",
            "question_id",
            "hf_source_file",
            "hf_row_group_idx",
            "hf_local_row_in_group",
            "question",
            "options_json",
            "gold_answer",
            "generated_answer",
            "generated_cot",
            "complexity_n_steps",
            "step_numbers_sequential",
            "topic_difficulty",
        }

        missing = required - set(table.columns)
        if missing:
            raise RuntimeError(
                f"Demonstration CSV is missing: {sorted(missing)}"
            )

        if len(table) != DEMO_EXPECTED_N:
            raise RuntimeError(
                f"Expected {DEMO_EXPECTED_N} demonstrations, "
                f"found {len(table)}."
            )

        if table["question_id"].astype(str).duplicated().any():
            raise RuntimeError(
                "Demonstration IDs are not unique."
            )

        self.table = table.reset_index(drop=True)
        print(
            "Validated demonstration bank:",
            self.csv_path,
            f"({len(self.table)} rows)",
        )

    def candidate_rows(
        self,
        subject: Optional[str],
        domain: str,
    ) -> pd.DataFrame:
        self._ensure_table()

        if subject in VALID_SUBJECTS:
            subset = self.table.loc[
                self.table["subject"].astype(str) == subject
            ].copy()
            scope = "subject"
        else:
            domain_subjects = set(DOMAIN_TO_SUBJECTS[domain])
            subset = self.table.loc[
                self.table["subject"].astype(str).isin(
                    domain_subjects
                )
            ].copy()
            scope = "domain"

        if len(subset) < SHOTS:
            raise RuntimeError(
                f"Too few demonstrations in {scope} pool."
            )

        subset["demo_selection_order"] = subset.index + 1
        subset["topic_difficulty"] = (
            subset["topic_difficulty"]
            .astype(str)
            .str.strip()
            .str.title()
        )
        subset["difficulty_priority"] = (
            subset["topic_difficulty"]
            .map(DIFFICULTY_PRIORITY)
        )

        if subset["difficulty_priority"].isna().any():
            raise RuntimeError(
                "Invalid demonstration difficulty label."
            )

        return subset

    def _ensure_source_files(self, relative_paths: list[str]):
        _paths = sorted(set(relative_paths))
        _missing_before = [
            rel for rel in _paths
            if not (DEMO_CACHE_DIR / rel).is_file()
        ]

        print(
            f"[DEMO FETCH START] requested={_paths} | "
            f"missing_before={_missing_before}",
            flush=True,
        )
        _fetch_t0 = time.perf_counter()

        if self.source_snapshot is None:
            self.source_snapshot = Path(
                snapshot_download(
                    repo_id=DEMO_REPO,
                    repo_type="dataset",
                    revision=DEMO_REVISION,
                    allow_patterns=_paths,
                    local_dir=str(DEMO_CACHE_DIR),
                )
            )
        else:
            snapshot_download(
                repo_id=DEMO_REPO,
                repo_type="dataset",
                revision=DEMO_REVISION,
                allow_patterns=_paths,
                local_dir=str(DEMO_CACHE_DIR),
            )

        _missing_after = [
            rel for rel in _paths
            if not (DEMO_CACHE_DIR / rel).is_file()
        ]
        print(
            f"[DEMO FETCH DONE] requested={_paths} | "
            f"elapsed_s={time.perf_counter() - _fetch_t0:.1f} | "
            f"missing_after={_missing_after}",
            flush=True,
        )

        if _missing_after:
            raise FileNotFoundError(
                "Demo source fetch completed but files are still missing: "
                + repr(_missing_after)
            )

    def load_demo(self, csv_row: pd.Series) -> dict:
        demo_id = str(csv_row["question_id"])

        if demo_id in self.demo_cache:
            print(
                f"[DEMO CACHE HIT] id={demo_id}",
                flush=True,
            )
            return self.demo_cache[demo_id]

        relative_path = str(csv_row["hf_source_file"])
        print(
            f"[DEMO LOAD START] id={demo_id} | source={relative_path}",
            flush=True,
        )
        _demo_t0 = time.perf_counter()

        self._ensure_source_files([relative_path])

        parquet_path = DEMO_CACHE_DIR / relative_path

        if not parquet_path.is_file():
            raise FileNotFoundError(
                f"Pinned demonstration source missing: {relative_path}"
            )

        if str(parquet_path) not in self.parquet_cache:
            self.parquet_cache[str(parquet_path)] = pq.ParquetFile(
                parquet_path
            )

        parquet_file = self.parquet_cache[str(parquet_path)]
        row_group_index = int(csv_row["hf_row_group_idx"])
        table = parquet_file.read_row_group(row_group_index)

        ids = [
            str(x)
            for x in table.column("id").to_pylist()
        ]
        local_index = int(csv_row["hf_local_row_in_group"])

        if (
            not 0 <= local_index < table.num_rows
            or ids[local_index] != demo_id
        ):
            matches = [
                i
                for i, value in enumerate(ids)
                if value == demo_id
            ]
            if len(matches) != 1:
                raise RuntimeError(
                    f"Could not uniquely resolve demonstration {demo_id}."
                )
            local_index = matches[0]

        source_row = table.slice(
            local_index,
            1,
        ).to_pylist()[0]

        if str(source_row["id"]) != demo_id:
            raise RuntimeError("Resolved the wrong demonstration row.")

        if str(source_row["question"]) != str(csv_row["question"]):
            raise RuntimeError(
                f"Question mismatch for demonstration {demo_id}."
            )

        source_options = parse_options_value(
            source_row["options"]
        )
        csv_options = parse_options_value(
            str(csv_row["options_json"])
        )

        if source_options != csv_options:
            raise RuntimeError(
                f"Option mismatch for demonstration {demo_id}."
            )

        gold = str(source_row["answer"]).strip().upper()

        if gold != str(csv_row["gold_answer"]).strip().upper():
            raise RuntimeError(
                f"Gold mismatch for demonstration {demo_id}."
            )

        if gold != str(csv_row["generated_answer"]).strip().upper():
            raise RuntimeError(
                f"Generated-answer mismatch for demonstration {demo_id}."
            )

        cot = str(csv_row["generated_cot"]).strip()
        if not cot or cot.lower() == "nan":
            raise RuntimeError(
                f"Missing CoT for demonstration {demo_id}."
            )

        if re.search(r"(?im)^\s*Answer\s*:", cot):
            raise RuntimeError(
                f"CoT contains an embedded Answer line: {demo_id}"
            )

        step_count = int(csv_row["complexity_n_steps"])
        parsed_steps = [
            int(x)
            for x in re.findall(
                r"(?im)^\s*Step\s+(\d+)\s*:",
                cot,
            )
        ]

        if parsed_steps != list(range(1, step_count + 1)):
            raise RuntimeError(
                f"Sequential STEP audit failed for {demo_id}."
            )

        image_values = []
        image_index = 1

        while True:
            key = f"image_{image_index}"

            if key not in source_row:
                break

            value = source_row.get(key)
            if value is not None:
                image_values.append(
                    image_to_pil(
                        value,
                        source_dir=parquet_path.parent,
                    )
                )

            image_index += 1

        if not image_values:
            raise RuntimeError(
                f"Demonstration has no image: {demo_id}"
            )

        demo_problem = build_problem(
            question=str(source_row["question"]),
            options=source_options,
            images=image_values,
            sample_id=demo_id,
            subject=str(csv_row["subject"]),
            difficulty=str(csv_row["topic_difficulty"]).strip().title(),
            source_dir=parquet_path.parent,
        )

        demo = {
            "sample_id": demo_id,
            "sample_order": int(csv_row.name) + 1,
            "subject": str(csv_row["subject"]),
            "topic_difficulty": str(
                csv_row["topic_difficulty"]
            ).strip().title(),
            "difficulty_priority": DIFFICULTY_PRIORITY[
                str(csv_row["topic_difficulty"]).strip().title()
            ],
            "complexity_n_steps": step_count,
            "generated_cot": cot,
            "gold_answer": gold,
            "problem": demo_problem,
        }

        self.demo_cache[demo_id] = demo
        print(
            f"[DEMO LOAD DONE] id={demo_id} | "
            f"images={len(image_values)} | "
            f"elapsed_s={time.perf_counter() - _demo_t0:.1f}",
            flush=True,
        )
        return demo

    def select(
        self,
        problem: Problem,
        subject: Optional[str],
        domain: str,
        retriever,
    ) -> tuple[list[dict], list[dict], str]:
        """
        Lazy P3 selection.

        Selection is resolved lexicographically:

          1) complexity_n_steps desc
          2) difficulty Hard > Medium > Easy
          3) cosine similarity ONLY if the Top-3 membership is still
             ambiguous at the selection boundary
          4) demo_selection_order asc
          5) demo_id asc

        Therefore CLIP query embeddings are not computed when STEP and
        difficulty alone already determine the three demonstration IDs.
        """

        rows = self.candidate_rows(
            subject,
            domain,
        )

        scope = (
            "subject"
            if subject in VALID_SUBJECTS
            else "domain"
        )

        # ----------------------------------------------------------
        # Stage 1 — metadata-only ranking.
        # No CLIP and no multimodal demo loading here.
        # ----------------------------------------------------------
        candidates = []

        for row_index, csv_row in rows.iterrows():
            candidates.append({
                "csv_index": row_index,
                "demo_id": str(
                    csv_row["question_id"]
                ),
                "demo_selection_order": int(
                    csv_row[
                        "demo_selection_order"
                    ]
                ),
                "complexity_n_steps": int(
                    csv_row[
                        "complexity_n_steps"
                    ]
                ),
                "topic_difficulty": str(
                    csv_row[
                        "topic_difficulty"
                    ]
                ).strip().title(),
                "difficulty_priority": int(
                    csv_row[
                        "difficulty_priority"
                    ]
                ),
                "text_score": None,
                "image_score": None,
                "total_score": None,
                "image_pair_count": None,
                "cosine_computed": False,
            })

        # Highest STEP first, then Hard > Medium > Easy.
        # Stable order is used only when cosine is not required.
        candidates.sort(
            key=lambda item: (
                -item[
                    "complexity_n_steps"
                ],
                -item[
                    "difficulty_priority"
                ],
                item[
                    "demo_selection_order"
                ],
                item[
                    "demo_id"
                ],
            )
        )

        selected = []
        cursor = 0
        query_embedding = None
        cosine_tiebreak_used = False
        cosine_tiebreak_group_size = 0
        cosine_tiebreak_slots = 0
        cosine_tiebreak_key = None

        while (
            len(selected) < SHOTS
            and cursor < len(candidates)
        ):
            first = candidates[cursor]

            group_key = (
                int(
                    first[
                        "complexity_n_steps"
                    ]
                ),
                int(
                    first[
                        "difficulty_priority"
                    ]
                ),
            )

            group = []
            while cursor < len(candidates):
                current = candidates[
                    cursor
                ]
                current_key = (
                    int(
                        current[
                            "complexity_n_steps"
                        ]
                    ),
                    int(
                        current[
                            "difficulty_priority"
                        ]
                    ),
                )

                if current_key != group_key:
                    break

                group.append(
                    current
                )
                cursor += 1

            remaining = (
                SHOTS
                - len(selected)
            )

            # Entire tied group fits inside the remaining Top-3 slots.
            # Its membership is already determined, so cosine is not needed.
            if len(group) <= remaining:
                selected.extend(
                    group
                )
                continue

            # ------------------------------------------------------
            # Stage 2 — cosine only for the boundary tie group.
            # This is the ONLY path that creates the current query
            # CLIP embedding.
            # ------------------------------------------------------
            cosine_tiebreak_used = True
            cosine_tiebreak_group_size = (
                len(group)
            )
            cosine_tiebreak_slots = (
                remaining
            )
            cosine_tiebreak_key = {
                "complexity_n_steps": (
                    group_key[0]
                ),
                "difficulty_priority": (
                    group_key[1]
                ),
            }

            if query_embedding is None:
                query_embedding = (
                    retriever.query_embedding(
                        problem
                    )
                )

            scored_group = []

            for item in group:
                score = (
                    retriever.similarity(
                        query_embedding,
                        item["demo_id"],
                    )
                )

                scored_item = {
                    **item,
                    **score,
                    "cosine_computed": True,
                }

                scored_group.append(
                    scored_item
                )

            scored_group.sort(
                key=lambda item: (
                    -float(
                        item[
                            "total_score"
                        ]
                    ),
                    int(
                        item[
                            "demo_selection_order"
                        ]
                    ),
                    str(
                        item[
                            "demo_id"
                        ]
                    ),
                )
            )

            selected.extend(
                scored_group[
                    :remaining
                ]
            )

        if len(selected) != SHOTS:
            raise RuntimeError(
                "Lazy P3 retrieval failed to select "
                f"{SHOTS} demonstrations."
            )

        print(
            f"[RICES SELECTED] sample={problem.sample_id} | "
            f"scope={scope} | cosine_tiebreak={cosine_tiebreak_used} | "
            f"demo_ids={[item['demo_id'] for item in selected]}",
            flush=True,
        )

        # ----------------------------------------------------------
        # Stage 3 — load only the three selected multimodal demos.
        # ----------------------------------------------------------
        selected_with_demos = []

        for item in selected:
            csv_row = self.table.loc[
                item[
                    "csv_index"
                ]
            ]

            demo = self.load_demo(
                csv_row
            )

            selected_with_demos.append({
                **item,
                "demo": demo,
            })

        # The selected list is already in descending P3 priority.
        # The prompt keeps the project's frozen reverse order:
        # rank 3 -> rank 2 -> rank 1.
        ranked = selected_with_demos
        prompt_order = list(
            reversed(
                ranked
            )
        )

        self.last_selection_audit = {
            "scope": scope,
            "candidate_count": (
                len(candidates)
            ),
            "cosine_tiebreak_used": (
                cosine_tiebreak_used
            ),
            "query_embedding_computed": (
                cosine_tiebreak_used
            ),
            "cosine_tiebreak_group_size": (
                cosine_tiebreak_group_size
            ),
            "cosine_tiebreak_slots": (
                cosine_tiebreak_slots
            ),
            "cosine_tiebreak_key": (
                cosine_tiebreak_key
            ),
            "selected_demo_ids": [
                item[
                    "demo_id"
                ]
                for item in ranked
            ],
        }

        return (
            [
                item["demo"]
                for item
                in prompt_order
            ],
            ranked,
            scope,
        )



print("Hybrid RICES retrieval loaded: runtime queries + frozen demos.")

Hybrid RICES retrieval loaded: runtime queries + frozen demos.


## 10. Strategy executors

In [11]:
def parse_final_answer(
    response_text: str,
    valid_letters: list[str],
) -> tuple[Optional[str], str]:
    lines = [
        line.strip()
        for line in str(response_text or "").splitlines()
        if line.strip()
    ]

    if not lines:
        return None, "empty_response"

    final_line = lines[-1]

    patterns = (
        ("plain", re.compile(
            r"^Answer:\s*([A-Z])\s*\.?$",
            re.IGNORECASE,
        )),
        ("dollar_prefix", re.compile(
            r"^Answer:\s*\$([A-Z])\s*\.?$",
            re.IGNORECASE,
        )),
        ("latex_math", re.compile(
            r"^Answer:\s*\$([A-Z])\$\s*\.?$",
            re.IGNORECASE,
        )),
        ("parenthesized", re.compile(
            r"^Answer:\s*\(([A-Z])\)\s*\.?$",
            re.IGNORECASE,
        )),
    )

    for name, pattern in patterns:
        match = pattern.fullmatch(final_line)

        if not match:
            continue

        letter = match.group(1).upper()

        if letter not in valid_letters:
            return None, "letter_outside_choice_set"

        return letter, f"final_line_{name}"

    return None, "unparseable_final_line"


def execute_cot(
    client: GoogleGemmaClient,
    problem: Problem,
) -> dict:
    text = (
        f"{normalize_marker_text(problem.question)}\n\n"
        f"{normalize_marker_text(format_options(problem.options))}\n\n"
        f"{COT_SUFFIX}"
    )

    parts = [
        PromptPart.image_part(image)
        for image in physical_images_in_occurrence_order(problem)
    ]
    parts.append(PromptPart.text_part(text))

    response = client.send(
        parts,
        purpose="cot_final",
        sample_id=problem.sample_id,
        max_tokens=MAX_OUTPUT_TOKENS,
    )

    prediction, parse_status = parse_final_answer(
        response["text"],
        problem.letters,
    )

    return {
        "strategy": "cot",
        "answer_letter": prediction,
        "answer_parse_status": parse_status,
        "response": response["text"],
        "api_trace": [response],
        "strategy_trace": {
            "prompt_sha256": response["prompt_sha256"],
            "image_occurrence_count": len(problem.occurrences),
        },
    }


def ccot_graph_targets(
    problem: Problem,
) -> list[ImageOccurrence]:
    signatures_by_ref = {}

    for occ in problem.occurrences:
        signature = (
            occ.source_section,
            occ.option_letter,
            occ.image_label,
            occ.context_placeholder,
            occ.graph_heading,
        )
        signatures_by_ref.setdefault(
            occ.image_ref,
            set(),
        ).add(signature)

    ambiguous = {
        ref: signatures
        for ref, signatures in signatures_by_ref.items()
        if len(signatures) != 1
    }

    if ambiguous:
        raise RuntimeError(
            "One physical image has multiple semantic roles, which "
            "is incompatible with the frozen one-graph-per-image CCoT "
            f"protocol: {ambiguous}"
        )

    targets = []
    seen = set()

    for occ in problem.occurrences:
        if occ.image_ref in seen:
            continue
        seen.add(occ.image_ref)
        targets.append(occ)

    return targets


def ccot_scene_graph_body(
    problem: Problem,
    target: ImageOccurrence,
) -> str:
    question_text, options_text = semantic_question_and_options(
        problem
    )

    if target.source_section == "question":
        return (
            f"{question_text}\n\n"
            f"{options_text}\n\n"
            f"{SCENE_GRAPH_INSTRUCTION}"
        )

    return (
        f"This image corresponds to Option {target.option_letter} "
        f"of the multiple-choice question below.\n\n"
        f"{question_text}\n\n"
        f"{options_text}\n\n"
        f"{SCENE_GRAPH_INSTRUCTION}"
    )


def execute_ccot(
    client: GoogleGemmaClient,
    problem: Problem,
) -> dict:
    graph_targets = ccot_graph_targets(problem)
    graph_by_ref = {}
    api_trace = []

    for target in graph_targets:
        parts = [
            PromptPart.text_part(
                f"[{target.image_label}]"
            ),
            PromptPart.image_part(
                problem.image_map[target.image_ref],
                label=target.image_label,
            ),
            PromptPart.text_part(
                ccot_scene_graph_body(
                    problem,
                    target,
                )
            ),
        ]

        response = client.send(
            parts,
            purpose=f"ccot_scene_graph_image_{target.image_ref}",
            sample_id=problem.sample_id,
            max_tokens=MAX_OUTPUT_TOKENS,
        )
        api_trace.append(response)
        graph_by_ref[target.image_ref] = response["text"]

    final_parts = []

    for occ in problem.occurrences:
        final_parts.extend([
            PromptPart.text_part(
                f"[{occ.image_label}]"
            ),
            PromptPart.image_part(
                problem.image_map[occ.image_ref],
                label=occ.image_label,
            ),
            PromptPart.text_part(
                f"{occ.graph_heading}\n"
                f"{graph_by_ref[occ.image_ref]}"
            ),
        ])

    question_text, options_text = semantic_question_and_options(
        problem
    )
    final_text = (
        "Use the images and their corresponding scene graphs as context "
        "and answer the following question:\n\n"
        f"{question_text}\n\n"
        f"{options_text}\n\n"
        f"{CCOT_FINAL_SUFFIX}"
    )
    final_parts.append(PromptPart.text_part(final_text))

    final_response = client.send(
        final_parts,
        purpose="ccot_final",
        sample_id=problem.sample_id,
        max_tokens=MAX_OUTPUT_TOKENS,
    )
    api_trace.append(final_response)

    prediction, parse_status = parse_final_answer(
        final_response["text"],
        problem.letters,
    )

    return {
        "strategy": "ccot",
        "answer_letter": prediction,
        "answer_parse_status": parse_status,
        "response": final_response["text"],
        "api_trace": api_trace,
        "strategy_trace": {
            "scene_graphs": {
                str(ref): text
                for ref, text in graph_by_ref.items()
            },
            "scene_graph_count": len(graph_targets),
            "final_prompt_sha256": final_response["prompt_sha256"],
        },
    }


def _read_p3_plan_checkpoints() -> dict[str, dict]:
    if not P3_PLAN_CHECKPOINT_PATH.exists():
        return {}

    records = {}
    for line_number, line in enumerate(
        P3_PLAN_CHECKPOINT_PATH.read_text(
            encoding="utf-8"
        ).splitlines(),
        start=1,
    ):
        if not line.strip():
            continue

        row = json.loads(line)
        sample_id = str(row["sample_id"])

        if sample_id in records:
            raise RuntimeError(
                "Duplicate P3 plan checkpoint for "
                f"{sample_id!r} at line {line_number}."
            )

        if not isinstance(row.get("plan_text"), str):
            raise RuntimeError(
                f"Invalid P3 plan checkpoint for {sample_id!r}."
            )

        records[sample_id] = row

    return records


def execute_plan_and_solve(
    client: GoogleGemmaClient,
    problem: Problem,
    *,
    request_pass: str = "auto",
) -> dict:
    """
    Frozen two-call P3 protocol.

    Cohort evaluation uses request_pass='main' and later 'deferred'.
    Interactive calls may use 'auto', which preserves the router's ordinary
    one-shot behavior but is not used by the frozen held-out scheduler.
    """
    if request_pass not in {"auto", "main", "deferred"}:
        raise ValueError(
            f"Invalid P3 request_pass={request_pass!r}."
        )

    question_text, options_text = semantic_question_and_options(
        problem
    )

    plan_parts = image_prefix_parts(
        problem,
        include_labels=True,
    )
    plan_parts.append(
        PromptPart.text_part(
            f"{question_text}\n\n"
            f"{options_text}\n\n"
            f"{PLAN_SUFFIX}"
        )
    )

    plan_records = _read_p3_plan_checkpoints()
    saved_plan = plan_records.get(problem.sample_id)
    plan_checkpoint_reused = saved_plan is not None

    if saved_plan is None:
        if request_pass == "auto":
            plan_response = client.send(
                plan_parts,
                purpose="plan_and_solve_plan",
                sample_id=problem.sample_id,
                max_tokens=P3_PLAN_MAX_OUTPUT_TOKENS,
            )
        else:
            plan_response = client.send_p3(
                plan_parts,
                purpose="plan_and_solve_plan",
                sample_id=problem.sample_id,
                max_tokens=P3_PLAN_MAX_OUTPUT_TOKENS,
                request_pass=request_pass,
            )

        raw_plan = plan_response["text"]

        # Call 1 is checkpointed immediately, before Call 2. This is the key
        # resume behavior of the original P3 notebook.
        saved_plan = {
            "sample_id": problem.sample_id,
            "plan_text": raw_plan,
            "plan_prompt_sha256": plan_response["prompt_sha256"],
            "plan_pass": plan_response["pass"],
            "plan_attempt": plan_response["attempt"],
            "plan_latency_s": plan_response["latency_s"],
            "prompt_tokens": plan_response["prompt_tokens"],
            "completion_tokens": plan_response["completion_tokens"],
            "total_tokens": plan_response["total_tokens"],
            "created_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "p3_scheduling_version": P3_SCHEDULING_VERSION,
        }
        append_jsonl(
            P3_PLAN_CHECKPOINT_PATH,
            saved_plan,
        )

        print(
            f"[P3 PLAN CHECKPOINTED] sample={problem.sample_id} | "
            f"pass={plan_response['pass']} | "
            f"tokens={plan_response['completion_tokens']}",
            flush=True,
        )
    else:
        raw_plan = saved_plan["plan_text"]
        # Reconstruct a compact API-trace placeholder. The provider call itself
        # is not repeated, exactly as in the original resume-safe notebook.
        plan_response = {
            "api_ok": True,
            "text": raw_plan,
            "latency_s": saved_plan.get("plan_latency_s"),
            "attempt": saved_plan.get("plan_attempt"),
            "pass": saved_plan.get("plan_pass"),
            "transport_mode": "inline_png",
            "prompt_sha256": saved_plan.get("plan_prompt_sha256"),
            "prompt_tokens": saved_plan.get("prompt_tokens"),
            "completion_tokens": saved_plan.get("completion_tokens"),
            "reasoning_tokens": None,
            "total_tokens": saved_plan.get("total_tokens"),
            "finish_reason": None,
            "response_id": None,
            "model_version": MODEL,
            "purpose": "plan_and_solve_plan",
            "checkpoint_reused": True,
        }
        print(
            f"[P3 PLAN RESUME] sample={problem.sample_id} | "
            "saved Call 1 reused; provider is NOT called again",
            flush=True,
        )

    final_parts = image_prefix_parts(
        problem,
        include_labels=True,
    )
    final_parts.append(
        PromptPart.text_part(
            f"{question_text}\n\n"
            f"{options_text}\n\n"
            f"Plan generated in the preceding call:\n"
            f"{raw_plan}\n\n"
            f"{PLAN_EXECUTION_SUFFIX}"
        )
    )

    if request_pass == "auto":
        final_response = client.send(
            final_parts,
            purpose="plan_and_solve_final",
            sample_id=problem.sample_id,
            max_tokens=P3_ANSWER_MAX_OUTPUT_TOKENS,
        )
    else:
        final_response = client.send_p3(
            final_parts,
            purpose="plan_and_solve_final",
            sample_id=problem.sample_id,
            max_tokens=P3_ANSWER_MAX_OUTPUT_TOKENS,
            request_pass=request_pass,
        )

    prediction, parse_status = parse_final_answer(
        final_response["text"],
        problem.letters,
    )

    return {
        "strategy": "plan_and_solve",
        "answer_letter": prediction,
        "answer_parse_status": parse_status,
        "response": final_response["text"],
        "api_trace": [
            plan_response,
            final_response,
        ],
        "strategy_trace": {
            "plan": raw_plan,
            "plan_checkpoint_reused": plan_checkpoint_reused,
            "p3_request_pass": request_pass,
            "p3_scheduling_version": P3_SCHEDULING_VERSION,
            "plan_prompt_sha256": plan_response["prompt_sha256"],
            "final_prompt_sha256": final_response["prompt_sha256"],
        },
    }


def execute_fewshot_complexity(
    client: GoogleGemmaClient,
    problem: Problem,
    subject: Optional[str],
    domain: str,
    demo_bank: DemoBank,
    retriever,
) -> dict:
    demos_in_prompt_order, ranked, retrieval_scope = demo_bank.select(
        problem=problem,
        subject=subject,
        domain=domain,
        retriever=retriever,
    )

    parts = [
        PromptPart.text_part(
            FEWSHOT_INSTRUCTION + "\n\n[DEMOS]\n\n"
        )
    ]

    for position, demo in enumerate(
        demos_in_prompt_order,
        start=1,
    ):
        parts.append(
            PromptPart.text_part(
                f"[DEMONSTRATION {position}]\n"
            )
        )
        append_problem_interleaved(
            parts,
            demo["problem"],
        )
        parts.append(
            PromptPart.text_part(
                f"\n[DEMONSTRATION {position} RESPONSE]\n"
                f"Reasoning:\n"
                f"{demo['generated_cot']}\n"
                f"Answer: {demo['gold_answer']}\n\n"
            )
        )

    parts.append(
        PromptPart.text_part("[FINAL QUERY]\n")
    )
    append_problem_interleaved(parts, problem)

    response = client.send(
        parts,
        purpose="fewshot_complexity_final",
        sample_id=problem.sample_id,
        max_tokens=MAX_OUTPUT_TOKENS,
    )

    prediction, parse_status = parse_final_answer(
        response["text"],
        problem.letters,
    )

    return {
        "strategy": "fewshot_complexity",
        "answer_letter": prediction,
        "answer_parse_status": parse_status,
        "response": response["text"],
        "api_trace": [response],
        "strategy_trace": {
            "retrieval_scope": retrieval_scope,
            "prompt_demo_ids": [
                demo["sample_id"]
                for demo in demos_in_prompt_order
            ],
            "ranked_demo_ids": [
                item["demo_id"]
                for item in ranked
            ],
            "ranked_demo_details": [
                {
                    "demo_id": item["demo_id"],
                    "complexity_n_steps": item[
                        "complexity_n_steps"
                    ],
                    "topic_difficulty": item[
                        "topic_difficulty"
                    ],
                    "difficulty_priority": item[
                        "difficulty_priority"
                    ],
                    "cosine_computed": item.get(
                        "cosine_computed",
                        False,
                    ),
                    "text_score": item.get(
                        "text_score"
                    ),
                    "image_score": item.get(
                        "image_score"
                    ),
                    "total_score": item.get(
                        "total_score"
                    ),
                    "image_pair_count": item.get(
                        "image_pair_count"
                    ),
                }
                for item in ranked
            ],
            "retrieval_audit": getattr(
                demo_bank,
                "last_selection_audit",
                None,
            ),
            "prompt_sha256": response["prompt_sha256"],
        },
    }


print("Strategy executors loaded.")

Strategy executors loaded.


## 13. Adaptive routing engine

The engine resolves the domain first, applies the frozen `DOMAIN_ROUTER`, and executes the selected strategy. Domain resolution is explicitly separated from answer generation in the returned audit trace.


In [12]:
class AdaptivePromptRouter:
    def __init__(
        self,
        api_key: Optional[str] = None,
        trained_classifier: Optional[TrainedClassifierProtocol] = None,
    ):
        self.client = GoogleGemmaClient(api_key=api_key)
        self.trained_classifier = trained_classifier
        self._demo_bank = None
        self._rices_retriever = None
        self._mmmu_resolver = None

    @property
    def demo_bank(self) -> DemoBank:
        if self._demo_bank is None:
            self._demo_bank = DemoBank()
        return self._demo_bank

    @property
    def rices_retriever(self) -> HybridRICESRetriever:
        if self._rices_retriever is None:
            self._rices_retriever = HybridRICESRetriever()
        return self._rices_retriever

    @property
    def mmmu_resolver(self) -> MMMUProResolver:
        if self._mmmu_resolver is None:
            self._mmmu_resolver = MMMUProResolver()
        return self._mmmu_resolver

    def _api_route(self, problem: Problem) -> dict:
        response = self.client.send(
            build_classifier_parts(problem),
            purpose="domain_classifier",
            sample_id=problem.sample_id,
            max_tokens=CLASSIFIER_MAX_OUTPUT_TOKENS,
        )
        parsed = parse_domain_classifier(response["text"])
        return {
            "domain": parsed["domain"],
            "subject": parsed["subject"],
            "domain_mode": "api_classifier",
            "domain_source": "gemma_api_classifier",
            "classifier_raw": response["text"],
            "classifier_parse_method": parsed["parse_method"],
            "classifier_metadata": None,
            "classifier_api_trace": response,
        }

    def _trained_route(self, problem: Problem) -> dict:
        if self.trained_classifier is None:
            raise RuntimeError(
                "domain_mode='trained_classifier' requires trained_classifier=..."
            )
        pil_images = [image_to_pil(asset.path) for asset in problem.images]
        pred = self.trained_classifier.predict(
            question=problem.question,
            images=pil_images,
            options=problem.options,
        )
        domain = normalize_label(pred.get("domain"))
        subject = normalize_label(pred.get("subject"))
        if domain not in VALID_DOMAINS or subject not in VALID_SUBJECTS:
            raise RuntimeError(
                f"Invalid trained-classifier output: domain={domain!r}, subject={subject!r}"
            )
        if SUBJECT_TO_DOMAIN[subject] != domain:
            raise RuntimeError(
                f"Inconsistent trained-classifier hierarchy: {domain!r}, {subject!r}"
            )
        return {
            "domain": domain,
            "subject": subject,
            "domain_mode": "trained_classifier",
            "domain_source": "trained_local_classifier",
            "classifier_raw": None,
            "classifier_parse_method": None,
            "classifier_metadata": {
                k: v for k, v in pred.items() if k not in {"domain", "subject"}
            },
            "classifier_api_trace": None,
        }

    def resolve_route(self, problem: Problem, domain_mode: str) -> dict:
        if domain_mode not in VALID_DOMAIN_MODES:
            raise ValueError(
                f"Unknown domain_mode={domain_mode!r}; choose {sorted(VALID_DOMAIN_MODES)}"
            )
        if domain_mode == "mmmu_pro":
            if problem.subject not in VALID_SUBJECTS or problem.domain not in VALID_DOMAINS:
                raise RuntimeError("MMMU-Pro mode requires validated benchmark metadata.")
            return {
                "domain": problem.domain,
                "subject": problem.subject,
                "domain_mode": "mmmu_pro",
                "domain_source": "pinned_mmmu_pro_subject_metadata",
                "classifier_raw": None,
                "classifier_parse_method": None,
                "classifier_metadata": None,
                "classifier_api_trace": None,
            }
        if domain_mode == "api_classifier":
            return self._api_route(problem)
        return self._trained_route(problem)

    def _run_strategy(
        self,
        problem: Problem,
        route: dict,
        *,
        p3_request_pass: str = "auto",
    ) -> dict:
        domain = route["domain"]
        subject = route["subject"]
        strategy = DOMAIN_ROUTER[domain]
        if strategy == "cot":
            return execute_cot(self.client, problem)
        if strategy == "ccot":
            return execute_ccot(self.client, problem)
        if strategy == "plan_and_solve":
            return execute_plan_and_solve(
                self.client,
                problem,
                request_pass=p3_request_pass,
            )
        if strategy == "fewshot_complexity":
            if subject not in VALID_SUBJECTS:
                raise RuntimeError("Few-shot Complexity requires a valid subject label.")
            return execute_fewshot_complexity(
                client=self.client,
                problem=problem,
                subject=subject,
                domain=domain,
                demo_bank=self.demo_bank,
                retriever=self.rices_retriever,
            )
        raise RuntimeError(f"Unsupported strategy: {strategy}")

    def infer(
        self,
        question: Optional[str] = None,
        images: Optional[Iterable[Any]] = None,
        options: Optional[Iterable[str]] = None,
        *,
        domain_mode: str = "api_classifier",
        sample_id: Optional[str] = None,
        p3_request_pass: str = "auto",
    ) -> dict:
        calls_before = self.client.call_counter

        if domain_mode == "mmmu_pro":
            if not sample_id:
                raise ValueError("MMMU-Pro mode requires sample_id.")
            problem = self.mmmu_resolver.problem(sample_id)

            # Optional user-supplied data are validation-only in benchmark mode.
            if question is not None:
                if options is None:
                    user_question, user_options = split_embedded_options(question)
                else:
                    user_question = str(question).strip()
                    user_options = [str(x).strip() for x in options]
                if user_question != problem.question or user_options != problem.options:
                    raise RuntimeError(
                        "Supplied question/options do not match the pinned MMMU-Pro sample."
                    )
            if images is not None:
                supplied = [ImageAsset.from_input(i + 1, x) for i, x in enumerate(list(images))]
                if [x.sha256 for x in supplied] != [x.sha256 for x in problem.images]:
                    raise RuntimeError("Supplied images do not match the pinned MMMU-Pro sample.")
        else:
            if question is None or images is None:
                raise ValueError("question and images are required outside MMMU-Pro mode.")
            problem = build_problem(
                question=question,
                images=images,
                options=options,
                sample_id=sample_id or "interactive",
            )

        if p3_request_pass not in {"auto", "main", "deferred"}:
            raise ValueError(
                f"Invalid p3_request_pass={p3_request_pass!r}."
            )

        route = self.resolve_route(problem, domain_mode)
        strategy_result = self._run_strategy(
            problem,
            route,
            p3_request_pass=p3_request_pass,
        )
        calls_after = self.client.call_counter

        result = {
            "sample_id": problem.sample_id,
            "domain_mode": route["domain_mode"],
            "domain_source": route["domain_source"],
            "domain": route["domain"],
            "subject": route["subject"],
            "strategy": strategy_result["strategy"],
            "answer": strategy_result["answer_letter"],
            "answer_parse_status": strategy_result["answer_parse_status"],
            "raw_response": strategy_result["response"],
            "classifier_raw": route.get("classifier_raw"),
            "classifier_parse_method": route.get("classifier_parse_method"),
            "classifier_metadata": route.get("classifier_metadata"),
            "api_calls": calls_after - calls_before,
            "model": MODEL,
            "api_key_source": self.client.api_key_source,
            "prompt_hashes": PROMPT_HASHES,
            "strategy_trace": strategy_result["strategy_trace"],
            "api_trace": (
                ([route["classifier_api_trace"]] if route.get("classifier_api_trace") else [])
                + strategy_result["api_trace"]
            ),
            "created_utc": datetime.now(timezone.utc).isoformat(),
        }

        append_jsonl(
            TRACE_PATH,
            {k: v for k, v in result.items() if k != "api_trace"},
        )
        return result


print("AdaptivePromptRouter ready with three domain modes.")

AdaptivePromptRouter ready with three domain modes.


## 14. One-line public product interface

The normal user-facing call is exactly:

```python
answer = solve(QUESTION, IMAGES, API_KEY)
```

The default is `domain_mode="api_classifier"`. Set `return_details=True` only when routing and inference provenance is needed.


In [13]:
_ROUTER_CACHE = {}


def _router_cache_key(api_key, trained_classifier):
    fingerprint = hashlib.sha256(
        str(api_key or "AUTO_RESOLVE").encode("utf-8")
    ).hexdigest()[:16]
    return fingerprint, (id(trained_classifier) if trained_classifier is not None else None)


def solve(
    question: Optional[str] = None,
    images: Optional[Iterable[Any]] = None,
    api_key: Optional[str] = None,
    *,
    options: Optional[Iterable[str]] = None,
    domain_mode: str = "api_classifier",
    sample_id: Optional[str] = None,
    trained_classifier: Optional[TrainedClassifierProtocol] = None,
    return_details: bool = False,
):
    """Public one-call interface; returns the parsed option letter by default."""
    if domain_mode == "trained_classifier" and trained_classifier is None:
        raise ValueError(
            "trained_classifier mode requires trained_classifier=..."
        )

    cache_key = _router_cache_key(api_key, trained_classifier)
    if cache_key not in _ROUTER_CACHE:
        _ROUTER_CACHE[cache_key] = AdaptivePromptRouter(
            api_key=api_key,
            trained_classifier=trained_classifier,
        )

    result = _ROUTER_CACHE[cache_key].infer(
        question=question,
        images=images,
        options=options,
        domain_mode=domain_mode,
        sample_id=sample_id,
    )
    return result if return_details else result["answer"]


print("Public interface ready: answer = solve(QUESTION, IMAGES, API_KEY)")

Public interface ready: answer = solve(QUESTION, IMAGES, API_KEY)


### Usage examples

**Ordinary user input**

```python
answer = solve(QUESTION, IMAGES, API_KEY)
```

**Trained-classifier mode**

```python
answer = solve(
    QUESTION, IMAGES, API_KEY,
    domain_mode="trained_classifier",
    trained_classifier=my_classifier,
)
```

**MMMU-Pro benchmark mode**

```python
answer = solve(
    api_key=API_KEY,
    domain_mode="mmmu_pro",
    sample_id="test_Physics_123",
)
```

When `<image n>` markers are present, their occurrence order controls image placement; repeated references remain repeated.


## 13. Structural unit tests — no API calls

These tests validate option extraction, marker ordering, routing-table integrity, prompt hashes, and the frozen answer parser before any paid or quota-limited request is sent.


# ------------------------------------------------------------
# Lazy P3 selection logic — metadata-only structural test
# ------------------------------------------------------------
def _lazy_selection_needs_cosine(
    groups,
    shots=3,
):
    selected = 0
    for group_size in groups:
        remaining = shots - selected
        if group_size <= remaining:
            selected += group_size
            if selected == shots:
                return False
        else:
            return True
    raise RuntimeError(
        "Insufficient synthetic candidates."
    )


# Top-3 identities are fixed by STEP+difficulty alone.
assert (
    _lazy_selection_needs_cosine(
        [1, 2, 5]
    )
    is False
)

# Boundary group contains more candidates than remaining slots.
assert (
    _lazy_selection_needs_cosine(
        [1, 4]
    )
    is True
)

# First priority group itself is larger than Top-3.
assert (
    _lazy_selection_needs_cosine(
        [5]
    )
    is True
)

print(
    "Lazy P3 boundary-selection tests: PASS"
)


# ------------------------------------------------------------
# CLIP feature-output compatibility tests
# ------------------------------------------------------------
import torch
from types import SimpleNamespace

_synthetic = torch.randn(2, CLIP_DIM)

# Historical Transformers behavior: direct Tensor.
_direct = extract_clip_projected_features(
    _synthetic,
    modality="text",
)
assert torch.equal(_direct, _synthetic)

# Newer Transformers behavior: BaseModelOutputWithPooling-like object.
_wrapped = SimpleNamespace(
    pooler_output=_synthetic
)
_unwrapped = extract_clip_projected_features(
    _wrapped,
    modality="image",
)
assert torch.equal(_unwrapped, _synthetic)

# Wrong hidden size must fail rather than silently entering cosine.
try:
    extract_clip_projected_features(
        torch.randn(1, CLIP_DIM + 1),
        modality="text",
    )
    raise AssertionError(
        "Expected dimension-mismatch failure."
    )
except RuntimeError:
    pass

print(
    "CLIP Tensor/BaseModelOutput compatibility tests: PASS"
)


In [14]:
# ------------------------------------------------------------
# Option parsing
# ------------------------------------------------------------
_test_stem, _test_options = split_embedded_options(
    """A visual question?
A. alpha
B. beta
C. gamma"""
)

assert _test_stem == "A visual question?"
assert _test_options == ["alpha", "beta", "gamma"]

# ------------------------------------------------------------
# Marker order + repetition
# ------------------------------------------------------------
_img = Image.new("RGB", (8, 8), "white")

_test_problem = build_problem(
    question="Compare <image 2> and <image 1> with <image 2>.",
    options=["x", "y"],
    images=[_img, _img],
    sample_id="unit_test",
)

assert [
    occ.image_ref
    for occ in _test_problem.occurrences
] == [2, 1, 2]

assert [
    image.index
    for image in physical_images_in_occurrence_order(
        _test_problem
    )
] == [2, 1, 2]

# ------------------------------------------------------------
# Routing table
# ------------------------------------------------------------
assert DOMAIN_ROUTER["Science"] == "ccot"
assert (
    DOMAIN_ROUTER["Health and Medicine"]
    == "fewshot_complexity"
)
assert (
    DOMAIN_ROUTER["Tech and Engineering"]
    == "plan_and_solve"
)

# ------------------------------------------------------------
# Answer parser
# ------------------------------------------------------------
letters = ["A", "B", "C", "D"]

assert parse_final_answer(
    "Reasoning\nAnswer: B",
    letters,
)[0] == "B"

assert parse_final_answer(
    "Reasoning\nAnswer: $C$",
    letters,
)[0] == "C"

assert parse_final_answer(
    "Reasoning only",
    letters,
)[0] is None

# ------------------------------------------------------------
# Prompt hashes
# ------------------------------------------------------------
assert PROMPT_HASHES["cot_suffix"] == EXPECTED_COT_SUFFIX_SHA256
assert (
    PROMPT_HASHES["scene_graph_instruction"]
    == EXPECTED_SCENE_GRAPH_INSTRUCTION_SHA256
)
assert (
    PROMPT_HASHES["plan_execution_suffix"]
    == EXPECTED_PLAN_EXECUTION_SUFFIX_SHA256
)
assert (
    PROMPT_HASHES["fewshot_instruction"]
    == EXPECTED_FEWSHOT_INSTRUCTION_SHA256
)

print("All structural unit tests PASSED.")

# Domain-mode contract
assert VALID_DOMAIN_MODES == {"mmmu_pro", "api_classifier", "trained_classifier"}
assert callable(solve)
print("Three-mode public-interface contract: PASS")


All structural unit tests PASSED.
Three-mode public-interface contract: PASS


## 16. Frozen 500-item evaluation adapter

The same held-out cohort can be evaluated in three scientifically distinct routing conditions: `mmmu_pro`, `api_classifier`, and `trained_classifier`. The resulting rows retain both answer accuracy and domain/subject classification accuracy.


## Frozen P3 scheduler audit

The Plan-and-Solve route below is intentionally synchronized with the original
Gemma 4 26B P3 notebook:

- Call 1 and Call 2 both use `max_output_tokens=8192`.
- Main pass: 1800-second request timeout, up to 3 attempts.
- Backoff: 15 seconds, then 30 seconds (capped at 90 seconds).
- At least 5 seconds between request start times.
- A main-pass failure is queued; it does **not** immediately enter the 2-hour pass.
- After the main pass reaches the end, exactly one deferred pass runs with
  7200-second request timeout and up to 3 attempts.
- Call 1 plans are checkpointed independently, so a successful plan is reused
  if Call 2 must be retried.
- P3 uses lossless inline PNG with the same 19 MiB safety gate.
- The raw visible plan is injected verbatim into Call 2.


In [15]:
assert P3_PLAN_MAX_OUTPUT_TOKENS == 8192
assert P3_ANSWER_MAX_OUTPUT_TOKENS == 8192
assert P3_REQUEST_TIMEOUT_S == 1800
assert P3_DEFERRED_REQUEST_TIMEOUT_S == 7200
assert P3_MAX_ATTEMPTS == 3
assert P3_DEFERRED_MAX_ATTEMPTS == 3
assert P3_DEFERRED_RETRY_PASSES == 1
assert P3_BASE_BACKOFF_S == 15.0
assert P3_MAX_BACKOFF_S == 90.0
assert P3_INTER_REQUEST_DELAY_S == 5.0
assert P3_RETRYABLE_STATUS == {408, 409, 429, 500, 502, 503, 504}
assert P3_INLINE_REQUEST_LIMIT_BYTES == 20 * 1024 * 1024
assert P3_INLINE_REQUEST_SAFETY_BYTES == int(19.0 * 1024 * 1024)
assert (
    P3_SCHEDULING_VERSION
    == "two_stage_per_request_pacing_defer_timeout_then_long_retry_v1"
)
print("Frozen P3 prompt/timing scheduler gate: PASS")

Frozen P3 prompt/timing scheduler gate: PASS


In [16]:
MMMU_PRO_REPO = "MMMU/MMMU_Pro"
MMMU_PRO_CONFIG = "standard (10 options)"
MMMU_PRO_SPLIT = "test"
MMMU_PRO_REVISION = (
    "563f3e84bb3b90893083a1f039cfa13077f2302b"
)

BENCHMARK_CONFIG = {
    "selected_ids_filename": "selected_500_ids.txt",
    "expected_n": 500,
    "source_n": 1730,
    "use_record_context": True,
}


def find_unique_named_file(
    filename: str,
    roots: tuple[Path, ...] = (
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ),
) -> Path:
    matches = sorted({
        path.resolve()
        for root in roots
        if root.exists()
        for path in root.rglob(filename)
        if path.is_file()
    })

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}."
        )

    if len(matches) > 1:
        contents = {
            hashlib.sha256(path.read_bytes()).hexdigest()
            for path in matches
        }

        if len(contents) != 1:
            raise RuntimeError(
                f"Conflicting copies of {filename} were found: "
                f"{matches}"
            )

    return matches[0]


def load_frozen_mmmupro():
    from datasets import load_dataset, Image as HFImage

    ids_path = find_unique_named_file(
        BENCHMARK_CONFIG["selected_ids_filename"]
    )
    selected_ids = [
        line.strip()
        for line in ids_path.read_text(
            encoding="utf-8-sig"
        ).splitlines()
        if line.strip()
    ]

    if (
        len(selected_ids) != BENCHMARK_CONFIG["expected_n"]
        or len(set(selected_ids))
        != BENCHMARK_CONFIG["expected_n"]
    ):
        raise RuntimeError(
            "Frozen benchmark ID list is not exactly 500 unique IDs."
        )

    source_ds = load_dataset(
        MMMU_PRO_REPO,
        MMMU_PRO_CONFIG,
        split=MMMU_PRO_SPLIT,
        revision=MMMU_PRO_REVISION,
    )

    if len(source_ds) != BENCHMARK_CONFIG["source_n"]:
        raise RuntimeError(
            f"Unexpected MMMU-Pro source size: {len(source_ds)}"
        )

    source_ids = [
        str(x)
        for x in source_ds["id"]
    ]
    index_by_id = {
        sample_id: i
        for i, sample_id in enumerate(source_ids)
    }

    missing = [
        sample_id
        for sample_id in selected_ids
        if sample_id not in index_by_id
    ]
    if missing:
        raise RuntimeError(
            f"Frozen IDs missing from pinned source: {missing[:10]}"
        )

    subset = source_ds.select(
        [
            index_by_id[sample_id]
            for sample_id in selected_ids
        ]
    )

    if [
        str(x)
        for x in subset["id"]
    ] != selected_ids:
        raise RuntimeError(
            "Frozen benchmark reconstruction order changed."
        )

    return subset, selected_ids


def benchmark_row_images(row: dict) -> list[Image.Image]:
    indexed = []

    for i in range(1, 64):
        key = f"image_{i}"
        if key not in row:
            break

        value = row.get(key)
        if value is not None:
            indexed.append((i, image_to_pil(value)))

    if not indexed:
        raise RuntimeError(
            f"No images found for benchmark row {row.get('id')}"
        )

    # Preserve physical image number so <image n> maps to images[n-1].
    expected = list(range(1, max(i for i, _ in indexed) + 1))
    observed = [i for i, _ in indexed]

    if observed != expected:
        raise RuntimeError(
            f"Non-contiguous benchmark image columns: {observed}"
        )

    return [image for _, image in indexed]


def read_completed_jsonl(path: Path) -> dict[str, dict]:
    if not path.exists():
        return {}

    rows = {}

    for line_number, line in enumerate(
        path.read_text(encoding="utf-8").splitlines(),
        start=1,
    ):
        if not line.strip():
            continue

        try:
            row = json.loads(line)
        except json.JSONDecodeError as exc:
            raise RuntimeError(
                f"Corrupt benchmark JSONL at line {line_number}: {exc}"
            ) from exc

        sample_id = str(row["sample_id"])
        if sample_id in rows:
            raise RuntimeError(
                f"Duplicate completed benchmark ID: {sample_id}"
            )

        rows[sample_id] = row

    return rows


def run_frozen_mmmupro(
    router: AdaptivePromptRouter,
    *,
    domain_mode: str = "mmmu_pro",
    output_path: Optional[Path] = None,
    max_items: Optional[int] = None,
) -> pd.DataFrame:
    """
    Held-out evaluator with exact original P3 scheduling for Plan-and-Solve.

    Non-P3 strategies retain their existing router behavior.
    P3 behavior:
      MAIN pass:     1800 s, up to 3 attempts/request.
      DEFERRED pass: one pass after MAIN reaches the end,
                     7200 s, up to 3 attempts/request.
      Call 1 and Call 2 are checkpointed independently.
      A successful Call 1 is never regenerated merely because Call 2 failed.
    """
    if domain_mode not in VALID_DOMAIN_MODES:
        raise ValueError(f"Invalid domain_mode: {domain_mode}")

    ds, selected_ids = load_frozen_mmmupro()

    if output_path is None:
        output_path = (
            WORKDIR
            / f"mmmu_pro_router_{domain_mode}.jsonl"
        )

    failure_path = output_path.with_name(
        output_path.stem + "_failures.jsonl"
    )

    completed = read_completed_jsonl(output_path)
    target_ids = (
        selected_ids
        if max_items is None
        else selected_ids[: int(max_items)]
    )

    run_started_mono = time.monotonic()
    deferred_ids = []
    consecutive_exhausted_429 = 0
    failed_this_run = 0
    soft_stop = False
    quota_stop = False
    fatal_stop = False
    stop_reason = None
    main_reached_end = True

    def wallclock_hours():
        return (
            time.monotonic() - run_started_mono
        ) / 3600.0

    def soft_stop_reached():
        hours = wallclock_hours()
        return (
            hours >= P3_SOFT_WALLCLOCK_HOURS,
            hours,
        )

    def persist_deferred_queue(phase: str, unresolved):
        P3_DEFERRED_QUEUE_PATH.write_text(
            json.dumps(
                {
                    "updated_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "phase": phase,
                    "deferred_sample_ids": list(
                        dict.fromkeys(unresolved)
                    ),
                    "successful_total": len(completed),
                    "target_total": len(target_ids),
                    "p3_plan_checkpoint_path": str(
                        P3_PLAN_CHECKPOINT_PATH
                    ),
                    "p3_scheduling_version": (
                        P3_SCHEDULING_VERSION
                    ),
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

    def persist_success(
        *,
        index: int,
        sample_id: str,
        row,
        result: dict,
        sample_started: float,
        phase: str,
    ):
        gold_subject = str(row["subject"])
        gold_domain = SUBJECT_TO_DOMAIN[
            gold_subject
        ]
        gold_answer = str(
            row["answer"]
        ).strip().upper()

        flat = {
            "sample_id": sample_id,
            "domain_mode": result["domain_mode"],
            "domain_source": result["domain_source"],
            "gold_subject": gold_subject,
            "predicted_subject": result["subject"],
            "subject_correct": (
                result["subject"] == gold_subject
            ),
            "gold_domain": gold_domain,
            "predicted_domain": result["domain"],
            "domain_correct": (
                result["domain"] == gold_domain
            ),
            "strategy": result["strategy"],
            "gold_answer": gold_answer,
            "predicted_answer": result["answer"],
            "correct": (
                result["answer"] == gold_answer
            ),
            "answer_parse_status": (
                result["answer_parse_status"]
            ),
            "api_calls": result["api_calls"],
            "raw_response": result["raw_response"],
            "classifier_raw": result[
                "classifier_raw"
            ],
            "created_utc": result["created_utc"],
            "scheduler_phase": phase,
        }

        append_jsonl(
            output_path,
            flat,
        )
        completed[sample_id] = flat

        sample_elapsed = (
            time.perf_counter()
            - sample_started
        )
        print(
            f"[SAMPLE DONE] {index + 1}/{len(target_ids)} | "
            f"id={sample_id} | phase={phase} | "
            f"strategy={result['strategy']} | "
            f"api_calls={result['api_calls']} | "
            f"answer={result['answer']} | "
            f"correct={flat['correct']} | "
            f"elapsed_s={sample_elapsed:.1f} | "
            f"completed={len(completed)}/{len(target_ids)}",
            flush=True,
        )

    def persist_failure(
        *,
        index: int,
        sample_id: str,
        row,
        expected_strategy: str,
        exc: Exception,
        sample_started: float,
        phase: str,
    ):
        nonlocal failed_this_run

        failed_this_run += 1
        status = status_from_exception(exc)
        sample_elapsed = (
            time.perf_counter()
            - sample_started
        )

        failure_record = {
            "sample_id": sample_id,
            "dataset_index": int(index),
            "domain_mode": domain_mode,
            "gold_subject": str(row["subject"]),
            "gold_domain": SUBJECT_TO_DOMAIN[
                str(row["subject"])
            ],
            "expected_strategy": expected_strategy,
            "scheduler_phase": phase,
            "api_ok": False,
            "http_status": status,
            "error_type": type(exc).__name__,
            "error_message": str(exc)[:8000],
            "elapsed_s": sample_elapsed,
            "created_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "p3_scheduling_version": (
                P3_SCHEDULING_VERSION
                if expected_strategy
                == "plan_and_solve"
                else None
            ),
        }
        append_jsonl(
            failure_path,
            failure_record,
        )

        print(
            f"[SAMPLE FAILED] {index + 1}/{len(target_ids)} | "
            f"id={sample_id} | phase={phase} | "
            f"strategy={expected_strategy} | "
            f"status={status} | "
            f"type={type(exc).__name__} | "
            f"elapsed_s={sample_elapsed:.1f} | "
            f"error={str(exc)[:300]}",
            flush=True,
        )
        return status

    print(
        f"[RUN START] mode={domain_mode} | "
        f"total={len(target_ids)} | "
        f"already_completed={len(completed)} | "
        f"output={output_path}",
        flush=True,
    )
    print(
        "[P3 SCHEDULER] "
        f"main={P3_REQUEST_TIMEOUT_S}s x "
        f"{P3_MAX_ATTEMPTS} attempts | "
        f"deferred={P3_DEFERRED_REQUEST_TIMEOUT_S}s x "
        f"{P3_DEFERRED_MAX_ATTEMPTS} attempts | "
        f"deferred_passes={P3_DEFERRED_RETRY_PASSES} | "
        f"inter_request_delay={P3_INTER_REQUEST_DELAY_S}s | "
        f"soft_wallclock={P3_SOFT_WALLCLOCK_HOURS}h",
        flush=True,
    )
    print(
        f"[P3 PLAN CHECKPOINT] "
        f"{P3_PLAN_CHECKPOINT_PATH}",
        flush=True,
    )
    print(
        f"[API EVENTS] {API_EVENTS_PATH}",
        flush=True,
    )
    print(
        f"[FAILURE LOG] {failure_path}",
        flush=True,
    )

    # ------------------------------------------------------------
    # MAIN PASS
    # ------------------------------------------------------------
    for index, sample_id in enumerate(
        tqdm(
            target_ids,
            desc=f"Heldout500 | {domain_mode} | MAIN",
        )
    ):
        if sample_id in completed:
            continue

        reached, hours = soft_stop_reached()
        if reached:
            soft_stop = True
            main_reached_end = False
            stop_reason = (
                f"soft_stop_{hours:.2f}h"
            )
            print(
                f"[SOFT STOP] {stop_reason}",
                flush=True,
            )
            break

        row = ds[index]
        gold_subject = str(row["subject"])
        gold_domain = SUBJECT_TO_DOMAIN[
            gold_subject
        ]
        expected_strategy = DOMAIN_ROUTER.get(
            gold_domain,
            "unknown",
        )

        sample_started = time.perf_counter()

        print(
            f"[SAMPLE START] "
            f"{index + 1}/{len(target_ids)} | "
            f"id={sample_id} | phase=MAIN | "
            f"domain={gold_domain} | "
            f"strategy={expected_strategy} | "
            f"completed={len(completed)}/{len(target_ids)}",
            flush=True,
        )

        try:
            kwargs = dict(
                domain_mode=domain_mode,
                sample_id=sample_id,
            )

            if domain_mode != "mmmu_pro":
                kwargs.update(
                    question=str(row["question"]),
                    options=parse_options_value(
                        row["options"]
                    ),
                    images=benchmark_row_images(row),
                )

            # Only Plan-and-Solve interprets this switch.
            kwargs["p3_request_pass"] = "main"

            result = router.infer(**kwargs)

        except Exception as exc:
            status = persist_failure(
                index=index,
                sample_id=sample_id,
                row=row,
                expected_strategy=expected_strategy,
                exc=exc,
                sample_started=sample_started,
                phase="MAIN",
            )

            if expected_strategy == "plan_and_solve":
                deferred_ids.append(sample_id)

                # Exact P3 quota-stop policy.
                if status == 429:
                    consecutive_exhausted_429 += 1
                else:
                    consecutive_exhausted_429 = 0

                if (
                    consecutive_exhausted_429
                    >= P3_MAX_CONSECUTIVE_EXHAUSTED_429
                ):
                    quota_stop = True
                    main_reached_end = False
                    stop_reason = (
                        "quota_stop_"
                        f"{consecutive_exhausted_429}"
                        "_consecutive_429"
                    )
                    persist_deferred_queue(
                        "MAIN",
                        deferred_ids,
                    )
                    print(
                        f"[QUOTA STOP] {stop_reason}",
                        flush=True,
                    )
                    break

                # A non-retryable provider status is fatal under the
                # original scheduler. Network/timeout/5xx/429 failures
                # remain deferred.
                if (
                    status is not None
                    and status not in P3_RETRYABLE_STATUS
                ):
                    fatal_stop = True
                    main_reached_end = False
                    stop_reason = (
                        f"fatal_MAIN_{status}_{sample_id}"
                    )
                    persist_deferred_queue(
                        "MAIN",
                        deferred_ids,
                    )
                    print(
                        f"[FATAL STOP] {stop_reason}",
                        flush=True,
                    )
                    break

            persist_deferred_queue(
                "MAIN",
                deferred_ids,
            )
            continue

        if expected_strategy == "plan_and_solve":
            consecutive_exhausted_429 = 0

        persist_success(
            index=index,
            sample_id=sample_id,
            row=row,
            result=result,
            sample_started=sample_started,
            phase="MAIN",
        )
        persist_deferred_queue(
            "MAIN",
            deferred_ids,
        )

    # ------------------------------------------------------------
    # ONE DEFERRED PASS — only after MAIN reaches the end cleanly.
    # ------------------------------------------------------------
    unresolved = list(
        dict.fromkeys(deferred_ids)
    )

    if (
        main_reached_end
        and not soft_stop
        and not quota_stop
        and not fatal_stop
        and P3_DEFERRED_RETRY_PASSES > 0
        and unresolved
    ):
        for pass_number in range(
            1,
            P3_DEFERRED_RETRY_PASSES + 1,
        ):
            next_unresolved = []

            for sample_id in tqdm(
                unresolved,
                desc=(
                    "Heldout500 | P3 | "
                    f"DEFERRED_{pass_number}"
                ),
            ):
                if sample_id in completed:
                    continue

                reached, hours = soft_stop_reached()
                if reached:
                    soft_stop = True
                    stop_reason = (
                        "soft_stop_deferred_"
                        f"{hours:.2f}h"
                    )
                    next_unresolved.extend(
                        sid
                        for sid in unresolved
                        if sid not in completed
                    )
                    print(
                        f"[SOFT STOP] {stop_reason}",
                        flush=True,
                    )
                    break

                index = selected_ids.index(
                    sample_id
                )
                row = ds[index]
                gold_subject = str(row["subject"])
                gold_domain = SUBJECT_TO_DOMAIN[
                    gold_subject
                ]
                expected_strategy = DOMAIN_ROUTER[
                    gold_domain
                ]

                # Defensive guard: the deferred queue is exclusively P3.
                if expected_strategy != "plan_and_solve":
                    raise RuntimeError(
                        "Non-P3 sample entered the P3 deferred queue: "
                        f"{sample_id} -> {expected_strategy}"
                    )

                sample_started = time.perf_counter()
                print(
                    f"[SAMPLE START] "
                    f"{index + 1}/{len(target_ids)} | "
                    f"id={sample_id} | "
                    f"phase=DEFERRED_{pass_number} | "
                    f"domain={gold_domain} | "
                    "strategy=plan_and_solve",
                    flush=True,
                )

                try:
                    kwargs = dict(
                        domain_mode=domain_mode,
                        sample_id=sample_id,
                        p3_request_pass="deferred",
                    )

                    if domain_mode != "mmmu_pro":
                        kwargs.update(
                            question=str(
                                row["question"]
                            ),
                            options=parse_options_value(
                                row["options"]
                            ),
                            images=benchmark_row_images(
                                row
                            ),
                        )

                    result = router.infer(**kwargs)

                except Exception as exc:
                    persist_failure(
                        index=index,
                        sample_id=sample_id,
                        row=row,
                        expected_strategy=expected_strategy,
                        exc=exc,
                        sample_started=sample_started,
                        phase=(
                            f"DEFERRED_{pass_number}"
                        ),
                    )
                    next_unresolved.append(
                        sample_id
                    )
                    persist_deferred_queue(
                        f"DEFERRED_{pass_number}",
                        next_unresolved,
                    )
                    continue

                persist_success(
                    index=index,
                    sample_id=sample_id,
                    row=row,
                    result=result,
                    sample_started=sample_started,
                    phase=(
                        f"DEFERRED_{pass_number}"
                    ),
                )
                persist_deferred_queue(
                    f"DEFERRED_{pass_number}",
                    next_unresolved,
                )

            unresolved = list(
                dict.fromkeys(next_unresolved)
            )

    persist_deferred_queue(
        "DONE" if not unresolved else "UNRESOLVED",
        unresolved,
    )

    successful_df = pd.DataFrame([
        completed[sample_id]
        for sample_id in target_ids
        if sample_id in completed
    ])

    print(
        f"[RUN DONE] mode={domain_mode} | "
        f"successful_total={len(successful_df)}/{len(target_ids)} | "
        f"failed_events_this_run={failed_this_run} | "
        f"p3_unresolved={len(unresolved)} | "
        f"soft_stop={soft_stop} | "
        f"quota_stop={quota_stop} | "
        f"fatal_stop={fatal_stop} | "
        f"stop_reason={stop_reason}",
        flush=True,
    )

    if unresolved:
        print(
            "[RESUME NOTE] P3 Call-1 checkpoints are retained. "
            "A later run will reuse saved plans and retry only the "
            "missing stage/final answer.",
            flush=True,
        )

    return successful_df


print("Three-mode Heldout500 adapter loaded.")

Three-mode Heldout500 adapter loaded.


## 15. Evaluation summary utilities

In [17]:
def summarize_benchmark(frame: pd.DataFrame) -> dict:
    if frame.empty:
        return {"n": 0, "answer_accuracy": None}
    return {
        "n": int(len(frame)),
        "answer_accuracy": float(frame["correct"].mean()),
        "domain_accuracy": float(frame["domain_correct"].mean()),
        "subject_accuracy": float(frame["subject_correct"].mean()),
        "parse_rate": float(frame["predicted_answer"].notna().mean()),
        "mean_api_calls_per_item": float(frame["api_calls"].mean()),
        "by_domain": frame.groupby("gold_domain")["correct"].agg(["count", "sum", "mean"]).reset_index().to_dict("records"),
        "by_strategy": frame.groupby("strategy")["correct"].agg(["count", "sum", "mean"]).reset_index().to_dict("records"),
    }


def display_benchmark_summary(frame: pd.DataFrame):
    summary = summarize_benchmark(frame)
    print("N:", summary["n"])
    if frame.empty:
        return
    print(f"Answer accuracy : {100*summary['answer_accuracy']:.2f}%")
    print(f"Domain accuracy : {100*summary['domain_accuracy']:.2f}%")
    print(f"Subject accuracy: {100*summary['subject_accuracy']:.2f}%")
    print(f"Mean API calls  : {summary['mean_api_calls_per_item']:.3f}")
    display(frame.groupby("gold_domain").agg(
        n=("sample_id", "size"),
        answer_accuracy=("correct", "mean"),
        domain_accuracy=("domain_correct", "mean"),
        subject_accuracy=("subject_correct", "mean"),
        mean_api_calls=("api_calls", "mean"),
    ).reset_index())


print("Evaluation summary utilities loaded.")

Evaluation summary utilities loaded.


## 18. Production Held-out-500 run — MMMU-Pro metadata routing + hybrid RICES

The final 500-item evaluation runs with:

```python
domain_mode = "mmmu_pro"
```

Domain resolution uses the published MMMU-Pro `subject` field and the frozen
subject-to-domain mapping; therefore no domain-classification API call is made.

### Required Kaggle inputs

1. `selected_500_ids.txt`;
2. `cot_pipeline_clean_merged_KEEP_only_grouped_by_subject.csv`;
3. the **original PASS RICES embedding output** that contains at least:
   - `demonstration_question_embeddings.parquet`
   - `demonstration_image_embeddings.parquet`
   - `embedding_manifest.json`
   - `FINAL_VALIDATION.json`

The historical query embeddings from the development experiment are ignored.

For a Health and Medicine sample, the router computes that sample's current
question/image CLIP embeddings at retrieval time, compares them against the
frozen 405 demonstration embeddings, applies the frozen P3 ranking, inserts the
selected three demonstrations, and then sends the final few-shot prompt to
Gemma 4 26B.

Checkpoint/resume behavior is unchanged.

In [18]:
# ================================================================
# FULL 500-SAMPLE PRODUCTION EVALUATION
# ================================================================

EVALUATION_DOMAIN_MODE = "mmmu_pro"
RUN_FULL_500 = True

assert EVALUATION_DOMAIN_MODE == "mmmu_pro"

RESULT_JSONL = (
    WORKDIR
    / "heldout500_mmmu_pro_results.jsonl"
)
RESULT_CSV = (
    WORKDIR
    / "heldout500_mmmu_pro_results.csv"
)
SUMMARY_JSON = (
    WORKDIR
    / "heldout500_mmmu_pro_summary.json"
)
DOMAIN_CSV = (
    WORKDIR
    / "heldout500_mmmu_pro_by_domain.csv"
)
SUBJECT_CSV = (
    WORKDIR
    / "heldout500_mmmu_pro_by_subject.csv"
)
STRATEGY_CSV = (
    WORKDIR
    / "heldout500_mmmu_pro_by_strategy.csv"
)


def preflight_mmmu_pro_500():
    if "DEMO_EXPECTED_N" not in globals():
        raise RuntimeError(
            "DEMO_EXPECTED_N is missing. Re-run the Few-shot "
            "demonstration configuration cell before the production preflight."
        )
    if int(DEMO_EXPECTED_N) != 405:
        raise RuntimeError(
            f"Unexpected demonstration-bank size invariant: {DEMO_EXPECTED_N}"
        )
    # ------------------------------------------------------------
    # 1) Frozen ID file
    # ------------------------------------------------------------
    ids_path = find_unique_named_file(
        BENCHMARK_CONFIG["selected_ids_filename"]
    )

    selected_ids = [
        line.strip()
        for line in ids_path.read_text(
            encoding="utf-8-sig"
        ).splitlines()
        if line.strip()
    ]

    if len(selected_ids) != 500:
        raise RuntimeError(
            f"Expected 500 selected IDs; found {len(selected_ids)}."
        )

    if len(set(selected_ids)) != 500:
        raise RuntimeError(
            "selected_500_ids.txt contains duplicate IDs."
        )

    # ------------------------------------------------------------
    # 2) Pinned MMMU-Pro reconstruction
    # ------------------------------------------------------------
    ds, recovered_ids = load_frozen_mmmupro()

    if len(ds) != 500:
        raise RuntimeError(
            f"Expected 500 reconstructed rows; found {len(ds)}."
        )

    if recovered_ids != selected_ids:
        raise RuntimeError(
            "Reconstructed MMMU-Pro order differs from selected_500_ids.txt."
        )

    subjects = [
        str(subject)
        for subject in ds["subject"]
    ]

    unknown_subjects = sorted(
        set(subjects) - set(SUBJECT_TO_DOMAIN)
    )
    if unknown_subjects:
        raise RuntimeError(
            f"Unmapped MMMU-Pro subjects: {unknown_subjects}"
        )

    if len(set(subjects)) != 30:
        raise RuntimeError(
            f"Expected all 30 subjects; observed {len(set(subjects))}."
        )

    domains = [
        SUBJECT_TO_DOMAIN[subject]
        for subject in subjects
    ]

    if set(domains) != set(DOMAIN_ROUTER):
        raise RuntimeError(
            "The selected cohort does not cover the complete six-domain taxonomy."
        )

    # ------------------------------------------------------------
    # 3) Demonstration-bank validation BEFORE any API request
    # ------------------------------------------------------------
    demo_path = discover_demo_csv()

    if sha_file(demo_path) != EXPECTED_DEMO_CSV_SHA256:
        raise RuntimeError(
            "Demonstration CSV SHA-256 mismatch."
        )

    demo_table = pd.read_csv(demo_path)

    if len(demo_table) != DEMO_EXPECTED_N:
        raise RuntimeError(
            f"Expected {DEMO_EXPECTED_N} demonstrations; "
            f"found {len(demo_table)}."
        )

    if demo_table["question_id"].astype(str).nunique() != DEMO_EXPECTED_N:
        raise RuntimeError(
            "Demonstration IDs are not unique."
        )

    missing_demo_subjects = sorted(
        set(VALID_SUBJECTS)
        - set(
            demo_table["subject"]
            .astype(str)
            .unique()
        )
    )
    if missing_demo_subjects:
        raise RuntimeError(
            "Demonstration bank lacks subjects: "
            f"{missing_demo_subjects}"
        )

    # ------------------------------------------------------------
    # 4) Route / strategy distribution
    # ------------------------------------------------------------
    audit = pd.DataFrame({
        "sample_id": selected_ids,
        "subject": subjects,
        "domain": domains,
    })
    audit["strategy"] = audit["domain"].map(
        DOMAIN_ROUTER
    )

    if audit["strategy"].isna().any():
        raise RuntimeError(
            "At least one domain has no routed strategy."
        )

    # Exact expected API-call count excluding provider retries.
    # CoT: 1
    # Few-shot Complexity: 1
    # Plan-and-Solve: 2
    # CCoT: physical_image_count + 1
    expected_calls = []

    for row in ds:
        subject = str(row["subject"])
        domain = SUBJECT_TO_DOMAIN[subject]
        strategy = DOMAIN_ROUTER[domain]

        if strategy in {
            "cot",
            "fewshot_complexity",
        }:
            calls = 1

        elif strategy == "plan_and_solve":
            calls = 2

        elif strategy == "ccot":
            image_count = len(
                benchmark_row_images(row)
            )
            calls = image_count + 1

        else:
            raise RuntimeError(
                f"Unexpected strategy: {strategy}"
            )

        expected_calls.append(calls)

    audit["expected_target_api_calls"] = (
        expected_calls
    )

    # ------------------------------------------------------------
    # 5) Historical demonstration embeddings — benchmark-critical
    # ------------------------------------------------------------
    rices_retriever = HybridRICESRetriever()

    # The preflight validates only the frozen demonstration side.
    # Query embeddings are intentionally computed at retrieval time.
    if (
        rices_retriever.demo_ids
        != demo_table["question_id"].astype(str).tolist()
    ):
        raise RuntimeError(
            "Historical demonstration embedding IDs differ from "
            "the frozen demonstration CSV."
        )

    if (
        rices_retriever.demo_text_meta["subject"]
        .astype(str)
        .tolist()
        != demo_table["subject"].astype(str).tolist()
    ):
        raise RuntimeError(
            "Historical demonstration embedding subjects differ "
            "from the frozen demonstration CSV."
        )

    if (
        rices_retriever.demo_text_meta["question_raw"]
        .astype(str)
        .tolist()
        != demo_table["question"].astype(str).tolist()
    ):
        raise RuntimeError(
            "Historical demonstration embedding questions differ "
            "from the frozen demonstration CSV."
        )

    print("=" * 88)
    print("MMMU-PRO HELD-OUT-500 PREFLIGHT: PASS")
    print("=" * 88)
    print("Selected IDs file :", ids_path)
    print("Selected rows     :", len(selected_ids))
    print("Unique IDs        :", len(set(selected_ids)))
    print("Subjects          :", len(set(subjects)))
    print("Domains           :", len(set(domains)))
    print("Demo CSV          :", demo_path)
    print("Demo rows         :", len(demo_table))
    print("Demo embeddings   :", rices_retriever.demo_embedding_root)
    print("Demo signature    :", rices_retriever.demo_bundle_signature)
    print("Query embeddings  :", "RUNTIME / per routed query")
    print("Demo semantic IDs :", "405 demos VERIFIED")
    print(
        "Expected target API calls "
        "(excluding retries):",
        int(audit["expected_target_api_calls"].sum()),
    )
    print()

    print("Domain distribution")
    display(
        audit["domain"]
        .value_counts()
        .rename_axis("domain")
        .reset_index(name="n")
        .sort_values("domain")
    )

    print("Strategy distribution")
    display(
        audit["strategy"]
        .value_counts()
        .rename_axis("strategy")
        .reset_index(name="n")
        .sort_values("strategy")
    )

    return {
        "ids_path": ids_path,
        "demo_path": demo_path,
        "selected_ids": selected_ids,
        "dataset": ds,
        "audit": audit,
        "rices_retriever": rices_retriever,
    }


PREFLIGHT = preflight_mmmu_pro_500()

if RUN_FULL_500:
    # API key is resolved by the router from:
    #   1) explicit argument (if supplied),
    #   2) Kaggle Secret GEMINI_API_KEY / GOOGLE_API_KEY,
    #   3) environment variable.
    #
    # No domain-classification API request is performed in mmmu_pro mode.
    router = AdaptivePromptRouter()
    router._rices_retriever = PREFLIGHT[
        "rices_retriever"
    ]

    benchmark_results = run_frozen_mmmupro(
        router,
        domain_mode="mmmu_pro",
        output_path=RESULT_JSONL,
        max_items=None,
    )

    # ------------------------------------------------------------
    # Completion gate
    # ------------------------------------------------------------
    if len(benchmark_results) > 500:
        raise RuntimeError(
            f"Unexpected result count: {len(benchmark_results)}"
        )

    complete = (
        len(benchmark_results) == 500
        and benchmark_results["sample_id"].nunique() == 500
    )

    # Always save the current checkpoint table.
    benchmark_results.to_csv(
        RESULT_CSV,
        index=False,
    )

    # ------------------------------------------------------------
    # Current metrics (final only when complete=True)
    # ------------------------------------------------------------
    if len(benchmark_results):
        by_domain = (
            benchmark_results
            .groupby("gold_domain")
            .agg(
                n=("sample_id", "size"),
                correct=("correct", "sum"),
                accuracy=("correct", "mean"),
                mean_api_calls=("api_calls", "mean"),
            )
            .reset_index()
            .sort_values("gold_domain")
        )

        by_subject = (
            benchmark_results
            .groupby("gold_subject")
            .agg(
                n=("sample_id", "size"),
                correct=("correct", "sum"),
                accuracy=("correct", "mean"),
                mean_api_calls=("api_calls", "mean"),
            )
            .reset_index()
            .sort_values("gold_subject")
        )

        by_strategy = (
            benchmark_results
            .groupby("strategy")
            .agg(
                n=("sample_id", "size"),
                correct=("correct", "sum"),
                accuracy=("correct", "mean"),
                mean_api_calls=("api_calls", "mean"),
            )
            .reset_index()
            .sort_values("strategy")
        )

        by_domain.to_csv(
            DOMAIN_CSV,
            index=False,
        )
        by_subject.to_csv(
            SUBJECT_CSV,
            index=False,
        )
        by_strategy.to_csv(
            STRATEGY_CSV,
            index=False,
        )

        accuracy = float(
            benchmark_results["correct"].mean()
        )
        parse_rate = float(
            benchmark_results[
                "predicted_answer"
            ].notna().mean()
        )
        mean_api_calls = float(
            benchmark_results[
                "api_calls"
            ].mean()
        )

    else:
        by_domain = pd.DataFrame()
        by_subject = pd.DataFrame()
        by_strategy = pd.DataFrame()
        accuracy = None
        parse_rate = None
        mean_api_calls = None

    summary = {
        "status": (
            "COMPLETE_HELDOUT500"
            if complete
            else "PROVISIONAL_PARTIAL"
        ),
        "domain_mode": "mmmu_pro",
        "domain_source": (
            "MMMU-Pro subject metadata -> frozen subject-to-domain mapping"
        ),
        "completed_rows": int(
            len(benchmark_results)
        ),
        "expected_rows": 500,
        "unique_completed_ids": int(
            benchmark_results["sample_id"].nunique()
            if len(benchmark_results)
            else 0
        ),
        "answer_accuracy": accuracy,
        "answer_parse_rate": parse_rate,
        "mean_api_calls_per_item": mean_api_calls,
        "domain_classifier_api_calls": 0,
        "fewshot_benchmark_retrieval": (
            "lazy_runtime_query_embeddings_plus_frozen_demo_embeddings"
        ),
        "embedding_protocol": EXPECTED_EMBEDDING_PROTOCOL,
        "demo_embedding_bundle_signature": PREFLIGHT[
            "rices_retriever"
        ].demo_bundle_signature,
        "demo_embedding_artifact_sha256": PREFLIGHT[
            "rices_retriever"
        ].demo_embedding_hashes,
        "query_embedding_timing": (
            "only_if_step_and_difficulty_leave_boundary_tie"
        ),
        "query_embedding_model": CLIP_REPO,
        "query_embedding_revision": CLIP_REVISION,
        "demo_csv_path": str(
            PREFLIGHT["demo_path"]
        ),
        "demo_csv_sha256": (
            EXPECTED_DEMO_CSV_SHA256
        ),
        "selected_ids_path": str(
            PREFLIGHT["ids_path"]
        ),
        "model": MODEL,
        "temperature": TEMPERATURE,
        "seed": API_SEED,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "thinking_level": THINKING_LEVEL,
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
    }

    SUMMARY_JSON.write_text(
        json.dumps(
            summary,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print()
    print("=" * 88)
    print(summary["status"])
    print("=" * 88)
    print(
        f"Completed: {summary['completed_rows']}/500"
    )

    if accuracy is not None:
        print(
            f"Accuracy : {100 * accuracy:.2f}%"
        )
        print(
            f"Parse rate: {100 * parse_rate:.2f}%"
        )
        print(
            f"Mean API calls/item: {mean_api_calls:.3f}"
        )

    print()
    print("Checkpoint JSONL:", RESULT_JSONL)
    print("Flat CSV        :", RESULT_CSV)
    print("Summary JSON    :", SUMMARY_JSON)

    if complete:
        print()
        print("By domain")
        display(by_domain)

        print("By strategy")
        display(by_strategy)

README.md: 0.00B [00:00, ?B/s]

standard (10 options)/test-00000-of-0000(…):   0%|          | 0.00/346M [00:00<?, ?B/s]

standard (10 options)/test-00001-of-0000(…):   0%|          | 0.00/332M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1730 [00:00<?, ? examples/s]

Frozen demonstration embeddings validated: /kaggle/input/notebooks/jingilifteyna/rices-embedding-construction/RICES_MMMU_MMMUPro_CLIP_ViTL14_embeddings_v1_1
Frozen demo embedding signature: d292fd0840b8f977af351689349d6aba37ecac90f0a0aba01b76ad7de90a0122
MMMU-PRO HELD-OUT-500 PREFLIGHT: PASS
Selected IDs file : /kaggle/input/notebooks/jingilifteyna/500-sample-mmmu-pro-held-out-test-cohort/mmmu_pro_heldout_500/selected_500_ids.txt
Selected rows     : 500
Unique IDs        : 500
Subjects          : 30
Domains           : 6
Demo CSV          : /kaggle/input/datasets/jingilifteyna/pooling/cot_pipeline_clean_merged_KEEP_only_grouped_by_subject.csv
Demo rows         : 405
Demo embeddings   : /kaggle/input/notebooks/jingilifteyna/rices-embedding-construction/RICES_MMMU_MMMUPro_CLIP_ViTL14_embeddings_v1_1
Demo signature    : d292fd0840b8f977af351689349d6aba37ecac90f0a0aba01b76ad7de90a0122
Query embeddings  : RUNTIME / per routed query
Demo semantic IDs : 405 demos VERIFIED
Expected target API 

...

[DEMO FETCH DONE] requested=['Pharmacy/test-00000-of-00001.parquet'] | elapsed_s=0.1 | missing_after=[]
[DEMO LOAD DONE] id=test_Pharmacy_307 | images=1 | elapsed_s=0.1
[API START] sample=test_Pharmacy_35 | purpose=fewshot_complexity_final | pass=main | attempt=1/3 | call_counter=96 | timeout_s=1800 | transport=inline_png
[API OK] sample=test_Pharmacy_35 | purpose=fewshot_complexity_final | pass=main | attempt=1/3 | elapsed_s=46.6 | prompt_tokens=2690 | completion_tokens=2178
[SAMPLE DONE] 58/500 | id=test_Pharmacy_35 | phase=MAIN | strategy=fewshot_complexity | api_calls=1 | answer=G | correct=True | elapsed_s=48.1 | completed=58/500
[SAMPLE START] 59/500 | id=test_Finance_7 | phase=MAIN | domain=Business | strategy=cot | completed=58/500
[API START] sample=test_Finance_7 | purpose=cot_final | pass=main | attempt=1/3 | call_counter=97 | timeout_s=1800 | transport=inline_png
[API OK] sample=test_Finance_7 | purpose=cot_final | pass=main | attempt=1/3 | elapsed_s=13.0 | prompt_tokens=69

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[API OK] sample=validation_Basic_Medical_Science_17 | purpose=fewshot_complexity_final | pass=main | attempt=1/3 | elapsed_s=4.8 | prompt_tokens=2057 | completion_tokens=171
[SAMPLE DONE] 86/500 | id=validation_Basic_Medical_Science_17 | phase=MAIN | strategy=fewshot_complexity | api_calls=1 | answer=E | correct=True | elapsed_s=4.9 | completed=86/500
[SAMPLE START] 87/500 | id=test_Diagnostics_and_Laboratory_Medicine_62 | phase=MAIN | domain=Health and Medicine | strategy=fewshot_complexity | completed=86/500
[RICES SELECTED] sample=test_Diagnostics_and_Laboratory_Medicine_62 | scope=subject | cosine_tiebreak=False | demo_ids=['test_Diagnostics_and_Laboratory_Medicine_26', 'test_Diagnostics_and_Laboratory_Medicine_100', 'test_Diagnostics_and_Laboratory_Medicine_149']
[DEMO CACHE HIT] id=test_Diagnostics_and_Laboratory_Medicine_26
[DEMO CACHE HIT] id=test_Diagnostics_and_Laboratory_Medicine_100
[DEMO CACHE HIT] id=test_Diagnostics_and_Laboratory_Medicine_149
[API START] sample=test_Dia

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[API OK] sample=test_Basic_Medical_Science_41 | purpose=fewshot_complexity_final | pass=main | attempt=1/3 | elapsed_s=6.3 | prompt_tokens=2098 | completion_tokens=246
[SAMPLE DONE] 118/500 | id=test_Basic_Medical_Science_41 | phase=MAIN | strategy=fewshot_complexity | api_calls=1 | answer=A | correct=True | elapsed_s=6.3 | completed=118/500
[SAMPLE START] 119/500 | id=validation_Pharmacy_21 | phase=MAIN | domain=Health and Medicine | strategy=fewshot_complexity | completed=118/500
[RICES SELECTED] sample=validation_Pharmacy_21 | scope=subject | cosine_tiebreak=False | demo_ids=['test_Pharmacy_390', 'test_Pharmacy_267', 'test_Pharmacy_307']
[DEMO CACHE HIT] id=test_Pharmacy_390
[DEMO CACHE HIT] id=test_Pharmacy_267
[DEMO CACHE HIT] id=test_Pharmacy_307
[API START] sample=validation_Pharmacy_21 | purpose=fewshot_complexity_final | pass=main | attempt=1/3 | call_counter=204 | timeout_s=1800 | transport=inline_png
[API OK] sample=validation_Pharmacy_21 | purpose=fewshot_complexity_final |

## 19. Interpreting the three domain modes

The three domain modes answer different routing questions:

- `mmmu_pro`: evaluates the frozen routing policy when the benchmark subject
  metadata is available;
- `api_classifier`: evaluates end-to-end routing when Gemma first predicts the
  domain and subject;
- `trained_classifier`: evaluates end-to-end routing with the project's trained
  domain/subject classifier.

Few-shot retrieval itself is held constant across these modes: the 405
demonstration embeddings are frozen from the original RICES pipeline, while the
current query embedding is generated at retrieval time with the same pinned
CLIP encoder and preprocessing protocol.

## 20. Suggested Methods wording

> **Adaptive domain-aware prompt routing.** We implemented a fixed multimodal
> prompt router over Gemma 4 26B A4B IT. Six academic domains were mapped to
> prompting strategies selected on the development cohort: CoT for Art and
> Design and Business, Few-shot Complexity for Health and Medicine,
> Plan-and-Solve for Humanities and Social Science and Tech and Engineering,
> and CCoT for Science. The mapping was frozen before held-out evaluation.
>
> For the final held-out-500 routing-policy evaluation, domain identity was
> obtained from MMMU-Pro subject metadata through a fixed subject-to-domain
> mapping. No domain-classification API call was made in this condition.
>
> **Few-shot Complexity used lazy hybrid RICES retrieval.** The question and
> image embeddings of the fixed 405-item MMMU demonstration bank were reused
> from the original validated RICES corpus. For each current query, candidate
> demonstrations were first restricted to the same subject and ordered by
> reasoning-step count and difficulty priority (`Hard > Medium > Easy`) without
> computing a query embedding. If these two criteria uniquely determined the
> three demonstration identities, no CLIP query encoding was performed. If the
> selection boundary remained tied, the current query was embedded at that
> point with the same pinned CLIP ViT-L/14 encoder and preprocessing protocol,
> and multimodal cosine similarity was computed only for the tied boundary
> group. Remaining ties were resolved by frozen demonstration order and
> demonstration ID. Historical development-query embeddings were never reused.
> Only the three selected multimodal demonstrations were then loaded for prompt
> construction.
>
> Runtime CLIP query encoding, when required, reproduced the original RICES
> semantic inputs and numerical settings: question-stem-only text after image
> marker removal, exclusion of options/answers/explanations/CoT, 77-token CLIP
> text context, EXIF-oriented RGB images, float32 inference, and unit-L2
> normalization.
>
> CoT, Plan-and-Solve, and CCoT retained their original controlled prompt
> templates and message assembly protocols.

## 21. Detailed output schema

`solve(...)` returns only the option letter by default. With `return_details=True`, the result includes `domain_mode`, `domain_source`, `domain`, `subject`, `strategy`, `answer`, classifier provenance, API call count, strategy trace, raw response, token/latency trace, and prompt hashes. The API key is never returned or serialized.


In [19]:
ROUTER_PROTOCOL = {
    "model": MODEL,
    "api_version": API_VERSION,
    "sdk_version": SDK_VERSION_PIN,
    "generation": {
        "temperature": TEMPERATURE,
        "top_p_sent": False,
        "top_k_sent": False,
        "seed": API_SEED,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "thinking_level": THINKING_LEVEL,
        "stream": STREAM,
    },
    "domain_modes": sorted(VALID_DOMAIN_MODES),
    "domain_router": DOMAIN_ROUTER,
    "subject_to_domain": SUBJECT_TO_DOMAIN,
    "prompt_hashes": PROMPT_HASHES,
    "fewshot_benchmark_retrieval": "lazy_runtime_query_embeddings_plus_frozen_demo_embeddings",
    "embedding_protocol": EXPECTED_EMBEDDING_PROTOCOL,
    "query_embedding_timing": "computed_only_if_step_and_difficulty_leave_boundary_tie",
    "demo_embedding_source": "historical_frozen_rices_artifacts",
    "cosine_scope": "boundary_tie_group_only",
    "pre_cosine_priority": "complexity_n_steps desc; difficulty Hard>Medium>Easy",
    "mmmu_pro": {
        "repo": MMMU_PRO_REPO,
        "config": MMMU_PRO_CONFIG,
        "split": MMMU_PRO_SPLIT,
        "revision": MMMU_PRO_REVISION,
    },
    "fewshot": {
        "shots": SHOTS,
        "demo_repo": DEMO_REPO,
        "demo_revision": DEMO_REVISION,
        "demo_csv_sha256": EXPECTED_DEMO_CSV_SHA256,
        "demo_expected_n": DEMO_EXPECTED_N,
        "clip_repo": CLIP_REPO,
        "clip_revision": CLIP_REVISION,
        "ranking": "steps desc; difficulty desc; multimodal cosine desc; stable order asc; demo ID asc",
        "prompt_order": "rank 3 -> rank 2 -> rank 1",
    },
}

ROUTER_PROTOCOL_SHA256 = hashlib.sha256(
    json.dumps(
        ROUTER_PROTOCOL,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

(WORKDIR / "router_protocol.json").write_text(
    json.dumps(
        {**ROUTER_PROTOCOL, "router_protocol_sha256": ROUTER_PROTOCOL_SHA256},
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Router protocol SHA-256:", ROUTER_PROTOCOL_SHA256)

Router protocol SHA-256: 1adcce26fac6ce6d9afaee74196a11ed1cb35e371a131f446a8d8814483d6862


## References

- Yue et al. **MMMU-Pro: A More Robust Multi-discipline Multimodal Understanding and Reasoning Benchmark.**
- MMMU-Pro repository: `MMMU-Benchmark/MMMU`
- Google Gen AI SDK: `googleapis/python-genai`
- OpenAI CLIP ViT-L/14 model artifact used for retrieval tie-breaking: `openai/clip-vit-large-patch14`

The strategy prompt strings in this notebook are frozen from the project's controlled Gemma 4 experiment implementations rather than rewritten for this router.